# 18 · Прогноз инцидентов по реестру: модуль + заглушка реестра (CPU)

**Accelerator = None**, интернет не нужен. Газовый bundle G2 не меняется.

Подтверждённых пожаров, подтоплений и НСД в данных нет; организаторы разрешили заглушки, если показано, что на реальных
данных решение заработает. Ноутбук:
- агрегирует панель 14 до объект × сутки (признаки на конец D);
- строит **заглушку реестра** на реальных сутках: инцидент тем вероятнее, чем сильнее сработали профильные датчики
  объекта в **сами сутки инцидента**; модель видит только данные ≤ D, окно прогноза [D+2; D+3);
- четыре заранее заданные силы связи: β = 0 (контроль — модель обязана НЕ обогнать правило), 1, 2 (основная), 3;
- обучает LightGBM + Platt по времени (train до 2025, калибровка 2025 H1, проверка 2025 H2), сравнивает с правилом
  «профильные тревоги за 7 суток», применяет пороги допуска, строит кривую обучения по числу инцидентов;
- выпускает `incident_head_bundle.zip` + `.sha256` (без pickle) и файлы заглушки для backend.

Цифры — **демо конвейера (evidence E4)**, не качество на реальных инцидентах. Контракт:
`docs/INCIDENT_REGISTER_CONTRACT_RU.md`.


## Как запустить

1. **Add Input** → выход ноутбука 14 (та же версия, что для 16/17).
2. **Accelerator → None**, `MODE = "FULL"` → **Save Version → Save & Run All**.
3. Скачать из Output: `results_18_incident_head.json`, `summary_18_incident_head_ru.md`,
   `incident_head_bundle.zip`, `incident_head_bundle.zip.sha256`, `incident_register_stub.csv`,
   `incident_register_coverage_stub.json`.


In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО ИЗМЕНЯТЬ
MODE = "FULL"   # "FULL" или "SMOKE" (1/4 объектов, проверка запуска)
print("Режим:", MODE)


In [ ]:
import os, warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy, pandas, polars, sklearn, lightgbm, sys
for m in (numpy, pandas, polars, sklearn, lightgbm):
    print(f"{m.__name__:10s}", m.__version__)
print("python    ", sys.version.split()[0])


## Встроенный код

Те же файлы, что в `ml/sensor_failure/`.


In [ ]:
# Исходный код (репозиторий на Kaggle не нужен)
SOURCES = {
"target_models_v3_runtime.py": "\"\"\"Ноутбук 15: сравнение моделей на целях панели v3 (T2a/T2b/T2c/T1a/T4) и typed-score контракт для бэкенда.\n\nПротокол зафиксирован до обучения (docs/EXPERIMENT_15_TARGETS_V3_RU.md):\n- только pre-2025, фолды fold_2023 / fold_2024 как в ноутбуке 05; 2021 (переход системы мониторинга, FAQ)\n  и 30 суток прогрева после него исключены;\n- purging по d_label_decision_end: метка обучающей строки решена до начала калибровки, калибровочной — до валидации;\n- модели: P0 априорная частота по типу датчика, R1 правило (недавние эпизоды), L1 логистическая регрессия,\n  G1 LightGBM, G2 LightGBM без календаря, G3 LightGBM без типа датчика/системы (проверка shortcut);\n- метрики: PR-AUC, ROC-AUC, Brier/ECE после Platt (калибровка на H1), бюджетная кривая top-k в сутки\n  с cooldown 72 ч и без; парный блочный бутстрэп по неделям для G1 против лучшего простого baseline;\n- выбор и evidence_level по заранее заданным gates; результат — typed-score контракт без построчных прогнозов.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport time\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nSCHEMA_VERSION = \"3.0\"\nTARGET_SPECS: dict[str, dict[str, Any]] = {\n    \"target_t2a_link_onset\": {\"tz\": \"T2a\", \"incident_type\": \"link_loss_onset\", \"family\": \"probability\",\n                              \"rule\": (\"f_link_onset_strict_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2b_link_sustained\": {\"tz\": \"T2b\", \"incident_type\": \"link_loss_sustained\",\n                                  \"family\": \"conditional_proxy_score\",\n                                  \"rule\": (\"f_link_completed_long_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2c_power_onset\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_onset\", \"family\": \"probability\",\n                               \"rule\": (\"f_power_onset_strict_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t2c_power_sustained\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_sustained\",\n                                   \"family\": \"conditional_proxy_score\",\n                                   \"rule\": (\"f_power_completed_long_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t1a_tech_value\": {\"tz\": \"T1a\", \"incident_type\": \"measurement_fault_onset\",\n                              \"family\": \"probability\",\n                              \"rule\": (\"f_n_tech_like_30d\", \"f_n_undefined_measurement_30d\"), \"competing\": None},\n    \"target_t4_gas_cross\": {\"tz\": \"T4\", \"incident_type\": \"gas_threshold_cross\", \"family\": \"probability\",\n                            \"rule\": (\"f_gas_max_7d\", \"f_gas_band_n_7d\"), \"competing\": \"competing_gas_cross_d1\"},\n}\nCATEGORICAL = [\"link_state\", \"power_state\", \"тип_датчика\", \"тип_инж_системы\"]\nCALENDAR = [\"d_weekday\", \"d_month\"]\nCATALOGUE = [\"тип_датчика\", \"тип_инж_системы\"]\nFORBIDDEN_PREFIXES = (\"target_\", \"reason_\", \"competing_\", \"d_label\", \"d_target\", \"d_channel\", \"d_object\", \"d_cutoff\")\nFOLDS = [\n    {\"name\": \"fold_2023\", \"train_end\": date(2023, 1, 1), \"calibration\": (date(2023, 1, 1), date(2023, 7, 1)),\n     \"validation\": (date(2023, 7, 1), date(2024, 1, 1))},\n    {\"name\": \"fold_2024\", \"train_end\": date(2024, 1, 1), \"calibration\": (date(2024, 1, 1), date(2024, 7, 1)),\n     \"validation\": (date(2024, 7, 1), date(2025, 1, 1))},\n]\nLOAD_END = date(2025, 1, 1)\nEXCLUDED = (date(2021, 1, 1), date(2022, 1, 31))   # 2021 + 30 суток прогрева окон признаков\n# selectable, но исход неизвестен: такие строки ранжируются в бюджете (слоты в unknown считаются), но не входят в PR-AUC\nUNKNOWN_REASONS = [\"unobserved_window\", \"ambiguous_onset\", \"censored_duration\", \"unknown_previous_reading\"]\nMODELS = [\"P0_type_prior\", \"R1_rule\", \"L1_logistic\", \"G1_lightgbm\", \"G2_no_calendar\", \"G3_no_catalogue\"]\nBASELINES_FOR_GATE = [\"P0_type_prior\", \"R1_rule\"]\n\n\ndef make_config_targets(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if mode not in (\"FULL\", \"SMOKE\"):\n        raise ValueError(\"MODE должен быть FULL или SMOKE\")\n    smoke = mode == \"SMOKE\"\n    cfg = {\n        \"mode\": mode,\n        \"input_dir\": os.environ.get(\"LDT_KAGGLE_INPUT_DIR\", \"/kaggle/input\"),\n        \"output_dir\": os.environ.get(\"LDT_OUTPUT_DIR\", \"/kaggle/working\"),\n        \"targets\": list(TARGET_SPECS),\n        \"smoke_channel_share\": 10,\n        \"verify_sha256\": True,\n        \"random_seed\": 42,\n        \"lgbm_iterations\": 60 if smoke else 300,\n        \"max_neg_per_pos\": 30,\n        \"max_train_rows\": 300_000 if smoke else 2_000_000,\n        \"max_logistic_rows\": 100_000 if smoke else 500_000,\n        \"budgets_per_day\": [5, 10, 25, 50],\n        \"cooldown_days\": 3,\n        \"bootstrap_reps\": 20 if smoke else 100,\n        \"min_val_positives\": 30,\n        \"shortcut_tolerance\": 0.10,\n        \"ece_calibrated_max\": 0.02,      # probability_calibrated только при ECE ≤ 0,02 в обоих фолдах\n        \"min_known_share_selectable\": 0.5,  # g6: в валидации исход известен хотя бы у половины selectable строк\n        \"max_unknown_share_calibrated\": 0.10,  # probability_calibrated: unknown среди selectable ≤ 10 %\n        \"min_type_positives\": 30,        # срез по типам датчиков: типы с ≥ 30 положительными\n    }\n    cfg.update(overrides or {})\n    return cfg\n\n\n# ---------------------------------------------------------------------------\n# Данные\n# ---------------------------------------------------------------------------\ndef file_sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        while block := f.read(1 << 22):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef discover_panel(input_dir: Path, verify: bool) -> tuple[list[Path], dict[str, Any]]:\n    found = sorted(Path(input_dir).rglob(\"panel_manifest_v3.json\"))\n    if not found:\n        raise FileNotFoundError(\"panel_manifest_v3.json не найден: подключите выход ноутбука 14 как Input\")\n    manifest_path = found[0]\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"schema_version\") != SCHEMA_VERSION:\n        raise ValueError(f\"ожидалась схема {SCHEMA_VERSION}, в manifest {manifest.get('schema_version')}\")\n    parts = []\n    for part in manifest[\"parts\"]:\n        p = manifest_path.parent / part[\"file\"]\n        if verify and file_sha256(p) != part[\"sha256\"]:\n            raise ValueError(f\"SHA-256 части {part['file']} не совпадает с manifest\")\n        parts.append(p)\n    import polars as pl\n\n    rows = sum(int(pl.scan_parquet(p).select(pl.len()).collect().item()) for p in parts)\n    if rows != manifest[\"rows\"]:\n        raise ValueError(f\"строк в частях {rows:,}, в manifest {manifest['rows']:,}\")\n    return parts, manifest\n\n\ndef safe_features(manifest: dict[str, Any], columns: list[str]) -> list[str]:\n    # тип датчика/системы — справочные метаданные канала (есть в панели, но не в списке признаков manifest v3)\n    feats = [c for c in dict.fromkeys(list(manifest[\"feature_columns\"]) + CATALOGUE) if c in columns]\n    bad = [c for c in feats if c.startswith(FORBIDDEN_PREFIXES)]\n    if bad:\n        raise ValueError(f\"запрещённые признаки: {bad}\")\n    return feats\n\n\ndef load_target(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any],\n                load_end: date = LOAD_END) -> Any:\n    \"\"\"load_end: граница загрузки (15 — всегда 2025-01-01; финальный экспорт 16 задаёт свою по рецепту).\"\"\"\n    reason = \"reason_\" + target[len(\"target_\"):]\n    lf = pl.scan_parquet(parts)\n    lf = lf.filter((pl.col(\"d_cutoff_date\") < load_end) & (pl.col(\"d_label_decision_end\") <= load_end)\n                   & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    comp = TARGET_SPECS[target].get(\"competing\")\n    extra = [comp] if comp and comp in lf.collect_schema().names() else []\n    cols = [\"d_channel_key\", \"d_cutoff_date\", \"d_label_decision_end\", target, reason] + extra + features\n    return (lf.filter(pl.col(target).is_not_null() | pl.col(reason).is_in(UNKNOWN_REASONS)).select(cols)\n            .with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect())\n\n\nSTRESS_PERIOD = (date(2021, 1, 1), date(2022, 1, 1))   # переход системы мониторинга (FAQ): только stress-срез\n\n\ndef load_stress(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any]) -> Any:\n    \"\"\"2021 — не участвует ни в обучении, ни в выборе; оценивается один раз моделями fold_2024 (GPT 20).\"\"\"\n    lf = pl.scan_parquet(parts).filter(pl.col(\"d_cutoff_date\").is_between(*STRESS_PERIOD, closed=\"left\")\n                                       & (pl.col(\"d_label_decision_end\") <= STRESS_PERIOD[1]) & pl.col(target).is_not_null())\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    return lf.select([target] + features).with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect()\n\n\ndef coverage(pl: Any, parts: list[Path], targets: list[str], cfg: dict[str, Any]) -> dict[str, Any]:\n    lf = pl.scan_parquet(parts).filter((pl.col(\"d_cutoff_date\") < LOAD_END)\n                                       & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    total = int(lf.select(pl.len()).collect().item())\n    out: dict[str, Any] = {\"rows_pre2025_without_2021\": total, \"targets\": {}}\n    for t in targets:\n        r = \"reason_\" + t[len(\"target_\"):]\n        g = lf.group_by(r).agg(pl.len()).collect()\n        reasons = {str(k): int(v) for k, v in g.iter_rows()}\n        labelled = reasons.get(\"positive\", 0) + reasons.get(\"negative\", 0) + reasons.get(\"negative_transient_only\", 0)\n        by_type = (lf.group_by(\"тип_датчика\").agg(pl.col(t).is_not_null().mean().alias(\"share\"), pl.len().alias(\"rows\"))\n                   .filter(pl.col(\"rows\") >= 1000).sort(\"rows\", descending=True).head(15).collect())\n        out[\"targets\"][t] = {\"labelled_rows\": labelled, \"labelled_share\": labelled / max(total, 1),\n                             \"reasons\": reasons,\n                             \"labelled_share_by_sensor_type\": {str(k or \"нет в справочнике\"): round(float(s), 4)\n                                                                for k, s, _ in by_type.iter_rows()}}\n    return out\n\n\ndef split_fold(pl: Any, frame: Any, fold: dict[str, Any]) -> tuple[Any, Any, Any]:\n    c0, c1 = fold[\"calibration\"]\n    v0, v1 = fold[\"validation\"]\n    d, e = pl.col(\"d_cutoff_date\"), pl.col(\"d_label_decision_end\")\n    train = frame.filter((d < fold[\"train_end\"]) & (e <= c0))\n    cal = frame.filter((d >= c0) & (d < c1) & (e <= v0))\n    val = frame.filter((d >= v0) & (d < v1) & (e <= v1))\n    return train, cal, val\n\n\ndef sample_train(pl: Any, train: Any, target: str, cfg: dict[str, Any]) -> Any:\n    pos = train.filter(pl.col(target) == 1)\n    neg = train.filter(pl.col(target) == 0)\n    n_neg = min(neg.height, max(pos.height * cfg[\"max_neg_per_pos\"], 1), max(cfg[\"max_train_rows\"] - pos.height, 1))\n    rate = n_neg / max(neg.height, 1)\n    neg = neg.sample(n=n_neg, seed=cfg[\"random_seed\"]) if n_neg < neg.height else neg\n    return pl.concat([pos.with_columns(pl.lit(1.0).alias(\"_w\")),\n                      neg.with_columns(pl.lit(1.0 / max(rate, 1e-9)).alias(\"_w\"))])\n\n\n# ---------------------------------------------------------------------------\n# Модели\n# ---------------------------------------------------------------------------\ndef _pandas(frame: Any, cols: list[str], levels: dict[str, list[str]]) -> Any:\n    import pandas as pd\n\n    x = pd.DataFrame({c: frame[c].to_numpy() for c in cols})\n    for c in cols:\n        if c in CATEGORICAL:\n            v = frame[c].fill_null(\"__NA__\").cast(str).to_numpy()\n            x[c] = pd.Categorical(v, categories=levels[c])\n    return x\n\n\nclass TypePrior:\n    def fit(self, train: Any, target: str, m: float = 50.0) -> \"TypePrior\":\n        import polars as pl\n\n        w = train[\"_w\"]\n        y = train[target].cast(float)\n        self.global_rate = float((w * y).sum() / w.sum())\n        g = train.with_columns(y.alias(\"_y\")).group_by(\"тип_датчика\").agg(\n            (pl.col(\"_w\") * pl.col(\"_y\")).sum().alias(\"p\"), pl.col(\"_w\").sum().alias(\"n\"))\n        self.rates = {k: (p + m * self.global_rate) / (n + m) for k, p, n in g.iter_rows()}\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return np.array([self.rates.get(k, self.global_rate) for k in frame[\"тип_датчика\"].to_list()], dtype=float)\n\n\ndef rule_score(frame: Any, spec: dict[str, Any]) -> np.ndarray:\n    a, b = spec[\"rule\"]\n    s = np.zeros(frame.height)\n    for col, w in ((a, 1.0), (b, 0.5)):\n        if col in frame.columns:\n            s = s + w * np.nan_to_num(frame[col].to_numpy().astype(float))\n    return s\n\n\nclass Logistic:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"Logistic\":\n        from sklearn.linear_model import LogisticRegression\n\n        if train.height > cfg[\"max_logistic_rows\"]:\n            train = train.sample(n=cfg[\"max_logistic_rows\"], seed=cfg[\"random_seed\"])\n        self.num = [c for c in features if c not in CATEGORICAL]\n        self.cat = {c: [str(v) for v in train[c].fill_null(\"__NA__\").cast(str).value_counts(sort=True)[c].head(50).to_list()]\n                    for c in features if c in CATEGORICAL}\n        x = self._x(train, fit=True)\n        self.model = LogisticRegression(C=1.0, max_iter=300, random_state=cfg[\"random_seed\"])\n        self.model.fit(x, train[target].to_numpy(), sample_weight=train[\"_w\"].to_numpy())\n        return self\n\n    def _x(self, frame: Any, fit: bool = False) -> np.ndarray:\n        num = np.column_stack([frame[c].to_numpy().astype(float) for c in self.num]) if self.num else np.zeros((frame.height, 0))\n        num = np.nan_to_num(num, nan=0.0, posinf=0.0, neginf=0.0)\n        num = np.sign(num) * np.log1p(np.abs(num))\n        if fit:\n            self.mu, self.sd = num.mean(0), num.std(0) + 1e-9\n        num = (num - self.mu) / self.sd\n        cats = [(frame[c].fill_null(\"__NA__\").cast(str).to_numpy()[:, None] == np.array(lv)[None, :]).astype(float)\n                for c, lv in self.cat.items()]\n        return np.hstack([num] + cats)\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(self._x(frame))[:, 1]\n\n\nclass LightGBM:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"LightGBM\":\n        from lightgbm import LGBMClassifier\n\n        self.features = features\n        self.levels = {c: sorted(str(v) for v in train[c].fill_null(\"__NA__\").cast(str).unique().to_list())\n                       for c in features if c in CATEGORICAL}\n        self.model = LGBMClassifier(objective=\"binary\", n_estimators=int(cfg[\"lgbm_iterations\"]), learning_rate=0.05,\n                                    num_leaves=63, max_bin=63, min_child_samples=100, subsample=0.8, subsample_freq=1,\n                                    colsample_bytree=0.8, random_state=cfg[\"random_seed\"], n_jobs=-1, verbosity=-1)\n        cats = [c for c in features if c in CATEGORICAL]\n        self.model.fit(_pandas(train, features, self.levels), train[target].to_numpy(),\n                       sample_weight=train[\"_w\"].to_numpy(), categorical_feature=cats or \"auto\")\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(_pandas(frame, self.features, self.levels))[:, 1]\n\n    def importance(self, top: int = 12) -> dict[str, float]:\n        gain = self.model.booster_.feature_importance(\"gain\")\n        total = float(gain.sum()) or 1.0\n        order = np.argsort(-gain)[:top]\n        return {self.features[i]: round(float(gain[i] / total), 4) for i in order}\n\n\n# ---------------------------------------------------------------------------\n# Метрики\n# ---------------------------------------------------------------------------\ndef _z(score: np.ndarray, is_probability: bool) -> np.ndarray:\n    if is_probability:\n        p = np.clip(score, 1e-6, 1 - 1e-6)\n        return np.log(p / (1 - p))\n    return np.log1p(np.maximum(score, 0))\n\n\ndef platt(raw_cal: np.ndarray, y_cal: np.ndarray, is_probability: bool) -> Any:\n    from sklearn.linear_model import LogisticRegression\n\n    if np.unique(y_cal).size < 2:\n        return None\n    m = LogisticRegression(max_iter=300)\n    m.fit(_z(raw_cal, is_probability).reshape(-1, 1), y_cal)\n    return m\n\n\ndef apply_platt(m: Any, raw: np.ndarray, is_probability: bool, prior: float) -> np.ndarray:\n    if m is None:\n        return np.full(len(raw), prior)\n    return m.predict_proba(_z(raw, is_probability).reshape(-1, 1))[:, 1]\n\n\ndef ece(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:\n    q = np.quantile(p, np.linspace(0, 1, bins + 1))\n    idx = np.clip(np.searchsorted(q, p, side=\"right\") - 1, 0, bins - 1)\n    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))\n\n\ndef budget_curve(days: np.ndarray, channels: np.ndarray, y: np.ndarray, score: np.ndarray,\n                 budgets: list[int], cooldown_days: int) -> dict[str, Any]:\n    \"\"\"top-k строк в сутки по всей selectable когорте; y: 1/0, −1 = исход неизвестен.\n    precision — нижняя граница (unknown = 0), precision_upper — верхняя (unknown = 1); recall — по известным положительным.\n    С cooldown канал не выдаётся повторно cooldown_days суток после алерта.\"\"\"\n    order = np.lexsort((-score, days))\n    d_sorted, c_sorted, y_sorted = days[order], channels[order], y[order]\n    starts = np.flatnonzero(np.r_[True, d_sorted[1:] != d_sorted[:-1]])\n    ends = np.r_[starts[1:], len(order)]\n    total_pos = int((y == 1).sum())\n    total_unk = int((y == -1).sum())\n    pos_sorted, unk_sorted = (y_sorted == 1).astype(int), (y_sorted == -1).astype(int)\n    out = {}\n    for k in budgets:\n        tp = n = unk = 0\n        tp_cd = n_cd = unk_cd = 0\n        last: dict[Any, int] = {}\n        for s, e in zip(starts, ends):\n            tp += int(pos_sorted[s:min(e, s + k)].sum())\n            unk += int(unk_sorted[s:min(e, s + k)].sum())\n            n += min(e - s, k)\n            day = int(d_sorted[s])\n            taken = 0\n            for i in range(s, e):\n                if taken >= k:\n                    break\n                ch = c_sorted[i]\n                if ch in last and day - last[ch] < cooldown_days:\n                    continue\n                last[ch] = day\n                taken += 1\n                tp_cd += int(pos_sorted[i])\n                unk_cd += int(unk_sorted[i])\n            n_cd += taken\n        out[str(k)] = {\"precision\": tp / max(n, 1), \"precision_upper\": (tp + unk) / max(n, 1), \"unknown_alerts\": unk,\n                       \"recall\": tp / max(total_pos, 1), \"alerts\": n,\n                       # recall по GPT 16: нижняя — все неизвестные вне бюджета положительные, верхняя — все выданные unknown положительные\n                       \"recall_low\": tp / max(total_pos + total_unk - unk, 1), \"recall_up\": (tp + unk) / max(total_pos + unk, 1),\n                       \"precision_cooldown\": tp_cd / max(n_cd, 1), \"precision_upper_cooldown\": (tp_cd + unk_cd) / max(n_cd, 1),\n                       \"unknown_alerts_cooldown\": unk_cd, \"recall_cooldown\": tp_cd / max(total_pos, 1),\n                       \"recall_low_cooldown\": tp_cd / max(total_pos + total_unk - unk_cd, 1),\n                       \"recall_up_cooldown\": (tp_cd + unk_cd) / max(total_pos + unk_cd, 1),\n                       \"alerts_cooldown\": n_cd}\n    return out\n\n\ndef evaluate(y: np.ndarray, raw: np.ndarray, cal: np.ndarray, budget_frame: tuple[np.ndarray, ...],\n             cfg: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Ранжирующие метрики и калибровка — по строкам с известным исходом; бюджет — по всей selectable когорте.\"\"\"\n    days, channels, y_all, raw_all = budget_frame\n    from sklearn.metrics import average_precision_score, roc_auc_score\n\n    ok = np.unique(y).size == 2\n    return {\n        \"pr_auc\": float(average_precision_score(y, raw)) if ok else None,\n        \"roc_auc\": float(roc_auc_score(y, raw)) if ok else None,\n        \"brier\": float(np.mean((cal - y) ** 2)),\n        \"ece10\": ece(y, cal),\n        \"budget\": budget_curve(days, channels, y_all, raw_all, cfg[\"budgets_per_day\"], cfg[\"cooldown_days\"]),\n    }\n\n\ndef by_type_pr_auc(y: np.ndarray, raw: np.ndarray, types: np.ndarray, min_pos: int) -> dict[str, float]:\n    from sklearn.metrics import average_precision_score\n\n    out = {}\n    for t in np.unique(types):\n        m = types == t\n        if y[m].sum() >= min_pos and y[m].min() != y[m].max():\n            out[str(t)] = float(average_precision_score(y[m], raw[m]))\n    return out\n\n\ndef paired_bootstrap(y: np.ndarray, a: np.ndarray, b: np.ndarray, weeks: np.ndarray, reps: int, seed: int) -> dict[str, Any]:\n    \"\"\"PR-AUC(a) − PR-AUC(b), блоки = календарные недели, одни и те же выборки для обеих моделей.\"\"\"\n    from sklearn.metrics import average_precision_score\n\n    uniq, inv = np.unique(weeks, return_inverse=True)\n    rng = np.random.default_rng(seed)\n    diffs = []\n    for _ in range(reps):\n        w = np.bincount(rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[inv].astype(float)\n        m = w > 0\n        if y[m].min() == y[m].max():\n            continue\n        diffs.append(average_precision_score(y[m], a[m], sample_weight=w[m])\n                     - average_precision_score(y[m], b[m], sample_weight=w[m]))\n    d = np.array(diffs)\n    if not len(d):\n        return {\"mean\": None, \"ci95_low\": None, \"ci95_high\": None, \"reps\": 0}\n    return {\"mean\": float(d.mean()), \"ci95_low\": float(np.percentile(d, 2.5)),\n            \"ci95_high\": float(np.percentile(d, 97.5)), \"reps\": int(len(d))}\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_target(pl: Any, frame: Any, target: str, features: list[str], cfg: dict[str, Any], log: Any,\n               stress: Any = None) -> dict[str, Any]:\n    spec = TARGET_SPECS[target]\n    res: dict[str, Any] = {\"folds\": {}}\n    for fold in FOLDS:\n        train, cal, val_all = split_fold(pl, frame, fold)\n        known = pl.col(target).is_not_null()\n        train, cal, val = train.filter(known), cal.filter(known), val_all.filter(known)\n        n = {\"train\": train.height, \"calibration\": cal.height, \"validation\": val.height,\n             \"train_pos\": int(train[target].sum()), \"calibration_pos\": int(cal[target].sum()),\n             \"validation_pos\": int(val[target].sum()), \"validation_selectable\": val_all.height,\n             \"validation_unknown\": val_all.height - val.height}\n        fr: dict[str, Any] = {\"rows\": n, \"models\": {}}\n        res[\"folds\"][fold[\"name\"]] = fr\n        if min(n[\"train_pos\"], n[\"calibration_pos\"]) < 5 or n[\"validation_pos\"] < 1:\n            fr[\"skipped\"] = \"слишком мало положительных для обучения/калибровки\"\n            continue\n        tr = sample_train(pl, train, target, cfg)\n        y_cal, y_val = cal[target].to_numpy().astype(int), val[target].to_numpy().astype(int)\n        days = np.array([(d - date(2019, 1, 1)).days for d in val[\"d_cutoff_date\"].to_list()])\n        weeks = days // 7\n        days_all = np.array([(d - date(2019, 1, 1)).days for d in val_all[\"d_cutoff_date\"].to_list()])\n        chans_all = val_all[\"d_channel_key\"].to_numpy()\n        y_all = val_all[target].fill_null(-1).to_numpy().astype(int)\n        known_mask = y_all >= 0\n        prior = float(train[target].mean())\n        raws: dict[str, np.ndarray] = {}\n        fitted: dict[str, Any] = {}\n        no_cal = [c for c in features if c not in CALENDAR]\n        no_cat = [c for c in features if c not in CATALOGUE]\n        for name in MODELS:\n            t0 = time.time()\n            if name == \"P0_type_prior\":\n                m = TypePrior().fit(tr, target)\n                is_p = True\n            elif name == \"R1_rule\":\n                m, is_p = None, False\n            elif name == \"L1_logistic\":\n                m, is_p = Logistic().fit(tr, target, features, cfg), True\n            else:\n                feats = {\"G1_lightgbm\": features, \"G2_no_calendar\": no_cal, \"G3_no_catalogue\": no_cat}[name]\n                m, is_p = LightGBM().fit(tr, target, feats, cfg), True\n            score = (lambda f: rule_score(f, spec)) if m is None else m.score\n            raw_cal, raw_all = score(cal), score(val_all)\n            raw_val = raw_all[known_mask]\n            cal_model = platt(raw_cal, y_cal, is_p)\n            p_val = apply_platt(cal_model, raw_val, is_p, prior)\n            raws[name] = raw_val\n            fitted[name] = m\n            fr[\"models\"][name] = evaluate(y_val, raw_val, p_val, (days_all, chans_all, y_all, raw_all), cfg)\n            fr[\"models\"][name][\"fit_s\"] = round(time.time() - t0, 1)\n        fr[\"prevalence_validation\"] = float(y_val.mean())\n        fr[\"unknown_share_selectable\"] = n[\"validation_unknown\"] / max(n[\"validation_selectable\"], 1)\n        comp = spec.get(\"competing\")\n        if comp and comp in val.columns:\n            from sklearn.metrics import average_precision_score\n            # C5 / GPT 16: estimand «любой onset в D+2» против «первого» (onset в D+1 — отдельный исход), строки не удаляются\n            c = val[comp].fill_null(False).to_numpy().astype(bool)\n            y_first = (y_val == 1) & ~c\n            fr[\"competing_d1\"] = {\"share_of_rows\": float(c.mean()), \"share_of_positives\": float(c[y_val == 1].mean()) if y_val.sum() else None,\n                                  \"pr_auc_first_onset\": {k: (float(average_precision_score(y_first, raws[k]))\n                                                             if y_first.any() else None) for k in MODELS}}\n        types = val[\"тип_датчика\"].fill_null(\"нет в справочнике\").to_numpy()\n        fr[\"by_sensor_type_pr_auc\"] = {k: by_type_pr_auc(y_val, raws[k], types, cfg[\"min_type_positives\"]) for k in MODELS}\n        fr[\"importance_G1\"] = fitted[\"G1_lightgbm\"].importance()\n        best_base = max(BASELINES_FOR_GATE, key=lambda k: fr[\"models\"][k][\"pr_auc\"] or -1)\n        fr[\"best_simple_baseline\"] = best_base\n        for cand in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\"):\n            fr[f\"bootstrap_{cand}_vs_{best_base}\"] = paired_bootstrap(\n                y_val, raws[cand], raws[best_base], weeks, cfg[\"bootstrap_reps\"], cfg[\"random_seed\"])\n        log(f\"  {fold['name']}: val {n['validation']:,} строк, 1: {n['validation_pos']:,}; PR-AUC \"\n            + \", \".join(f\"{k.split('_')[0]} {fr['models'][k]['pr_auc']:.3f}\" for k in MODELS if fr['models'][k]['pr_auc'] is not None))\n    res[\"decision\"] = decide(res, cfg)\n    last = res[\"folds\"].get(FOLDS[-1][\"name\"], {})\n    if stress is not None and stress.height and \"skipped\" not in last and last.get(\"models\"):\n        from sklearn.metrics import average_precision_score\n        y_s = stress[target].to_numpy().astype(int)\n        out = {\"rows\": stress.height, \"positives\": int(y_s.sum()), \"prevalence\": float(y_s.mean()), \"pr_auc\": {}}\n        if 0 < y_s.sum() < len(y_s):\n            for name, m in fitted.items():\n                raw = rule_score(stress, spec) if m is None else m.score(stress)\n                out[\"pr_auc\"][name] = float(average_precision_score(y_s, raw))\n        res[\"stress_2021\"] = out\n    return res\n\n\ndef decide(res: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    folds = list(res[\"folds\"].values())\n    gates: dict[str, Any] = {}\n    gates[\"g1_enough_validation_events\"] = all(f[\"rows\"][\"validation_pos\"] >= cfg[\"min_val_positives\"] and \"skipped\" not in f\n                                               for f in folds)\n    if not gates[\"g1_enough_validation_events\"]:\n        return {\"gates\": gates, \"selected\": None, \"evidence_level\": \"insufficient_events\",\n                \"abstain_reason\": \"too_few_validation_events\"}\n\n    def beats(model: str) -> bool:\n        return all((f[f\"bootstrap_{model}_vs_{f['best_simple_baseline']}\"][\"ci95_low\"] or -1) > 0 for f in folds)\n\n    def ratio(model: str) -> float:\n        return min(f[\"models\"][model][\"pr_auc\"] / max(f[\"models\"][\"G1_lightgbm\"][\"pr_auc\"], 1e-12) for f in folds)\n\n    tol = 1 - cfg[\"shortcut_tolerance\"]\n    gates[\"g2_G1_beats_simple_baselines_both_folds\"] = beats(\"G1_lightgbm\")\n    gates[\"g3_no_calendar_shortcut\"] = ratio(\"G2_no_calendar\") >= tol\n    gates[\"g4_no_catalogue_shortcut\"] = ratio(\"G3_no_catalogue\") >= tol\n    base_means = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in BASELINES_FOR_GATE}\n\n    def stable(model: str) -> bool | None:\n        \"\"\"≥ 2/3 типов датчиков с поддержкой: модель лучше лучшего простого baseline в каждом фолде.\n        Неприменимо (None), если в фолдах меньше двух типов с поддержкой (например, газовая цель T4).\"\"\"\n        applicable = False\n        for f in folds:\n            base = f[\"by_sensor_type_pr_auc\"][f[\"best_simple_baseline\"]]\n            mine = f[\"by_sensor_type_pr_auc\"][model]\n            common = [k for k in mine if k in base]\n            if len(common) < 2:\n                continue\n            applicable = True\n            if sum(mine[k] > base[k] for k in common) < 2 / 3 * len(common):\n                return False\n        return True if applicable else None\n\n    gates[\"g5_logistic_beats_simple_baselines_both_folds\"] = beats(\"L1_logistic\")\n    mean_pr = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\")}\n    if gates[\"g2_G1_beats_simple_baselines_both_folds\"]:\n        # если без календаря модель тоже побеждает baseline, а календарь даёт подозрительно много — берём G2\n        selected = \"G2_no_calendar\" if (not gates[\"g3_no_calendar_shortcut\"] and beats(\"G2_no_calendar\")) else \"G1_lightgbm\"\n        # при равенстве (±0,005 PR-AUC) предпочитаем более простую и устойчивую логистическую регрессию\n        if gates[\"g5_logistic_beats_simple_baselines_both_folds\"] and mean_pr[\"L1_logistic\"] >= mean_pr[selected] - 0.005:\n            selected = \"L1_logistic\"\n        level = \"validated_proxy_model\"\n    elif gates[\"g5_logistic_beats_simple_baselines_both_folds\"]:\n        selected, level = \"L1_logistic\", \"validated_proxy_model\"\n    else:\n        selected = max(base_means, key=base_means.get)\n        level = \"baseline_only\"\n    gates[\"g6_known_outcome_share\"] = all(1 - f[\"unknown_share_selectable\"] >= cfg[\"min_known_share_selectable\"] for f in folds)\n    gates[\"g7_stable_across_sensor_types\"] = stable(selected) if level == \"validated_proxy_model\" else None\n    return {\"gates\": gates, \"selected\": selected, \"evidence_level\": level, \"abstain_reason\": None,\n            \"simple_baseline_mean_pr_auc\": base_means}\n\n\ndef score_kind(spec: dict[str, Any], decision: dict[str, Any], folds: dict[str, Any], cfg: dict[str, Any]) -> str:\n    \"\"\"Что отдаёт сервис: калиброванную вероятность обещаем только при подтверждённой модели и ECE ≤ порога.\"\"\"\n    if decision[\"evidence_level\"] == \"insufficient_events\":\n        return \"abstain\"\n    if spec[\"family\"] == \"conditional_proxy_score\":\n        return \"conditional_proxy_score\"  # цензура длительности: на всю популяцию не калибруем (согласовано 10/Q1)\n    sel = decision[\"selected\"]\n    eces = [f[\"models\"][sel][\"ece10\"] for f in folds.values() if sel in f.get(\"models\", {})]\n    unknown = [f.get(\"unknown_share_selectable\", 1.0) for f in folds.values()]\n    # ECE на наблюдаемых строках не доказывает калибровку для всех каналов: требуем ещё малую долю unknown (GPT 14)\n    if (decision[\"evidence_level\"] == \"validated_proxy_model\" and eces and max(eces) <= cfg[\"ece_calibrated_max\"]\n            and max(unknown) <= cfg[\"max_unknown_share_calibrated\"]):\n        return \"probability_calibrated\"\n    return \"proxy_score\"\n\n\ndef backend_contract(results: dict[str, Any], cov: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    out = {\"schema_version\": SCHEMA_VERSION, \"mode\": cfg[\"mode\"],\n           \"window\": \"метка: события в [D+2; D+3) по признакам на конец D; решение метки не позже D+4\",\n           \"cooldown_days\": cfg[\"cooldown_days\"], \"targets\": [],\n           \"data_quality_flags\": [{\"tz_item\": \"T1b\", \"incident_type\": \"clock_error_1970\", \"score_kind\": \"data_quality_flag\",\n                                   \"source\": \"значение 01.01.1970 в журнале (FAQ: «сбилась дата»); признаки f_n_clock_1970_*\",\n                                   \"not_a_failure_label\": True}],\n           \"evidence_levels\": {\"validated_proxy_model\": \"модель лучше простых baseline в обоих фолдах (95% CI)\",\n                               \"baseline_only\": \"отдаётся скор простого правила/приора\",\n                               \"insufficient_events\": \"мало событий — abstain\"}}\n    for t, r in results.items():\n        spec = TARGET_SPECS[t]\n        d = r[\"decision\"]\n        last = r[\"folds\"].get(\"fold_2024\", {})\n        sel = d.get(\"selected\")\n        m = last.get(\"models\", {}).get(sel, {}) if sel else {}\n        reasons = cov[\"targets\"][t][\"reasons\"]\n        abstain = {k: v for k, v in sorted(reasons.items(), key=lambda kv: -kv[1])\n                   if k not in (\"positive\", \"negative\", \"negative_transient_only\")}\n        entry = {\n            \"target\": t, \"tz_item\": spec[\"tz\"], \"incident_type\": spec[\"incident_type\"],\n            \"score_kind\": score_kind(spec, d, r[\"folds\"], cfg),\n            \"evidence_level\": d[\"evidence_level\"], \"selected_model\": sel,\n            \"coverage_labelled_share\": round(cov[\"targets\"][t][\"labelled_share\"], 4),\n            \"abstain_reasons_rows\": abstain,\n            \"validation_fold_2024\": ({\"pr_auc_on_known\": m.get(\"pr_auc\"), \"prevalence_on_known\": last.get(\"prevalence_validation\"),\n                                      \"unknown_share_selectable\": last.get(\"unknown_share_selectable\"),\n                                      \"brier\": m.get(\"brier\"), \"ece10\": m.get(\"ece10\"),\n                                      \"budget_25_per_day\": m.get(\"budget\", {}).get(\"25\")} if m else None),\n            \"not_a_confirmed_failure\": True,\n        }\n        if spec[\"family\"] == \"conditional_proxy_score\":\n            lab = cov[\"targets\"][t][\"labelled_rows\"]\n            cen = reasons.get(\"censored_duration\", 0)\n            entry[\"censoring\"] = {\"censored_rows\": cen, \"censored_share_of_decidable\": cen / max(lab + cen, 1),\n                                  \"note\": \"метрики — по нецензурированным строкам; для всей популяции это bounds, не оценка\"}\n        out[\"targets\"].append(entry)\n    return out\n\n\ndef run_targets_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t_start = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    parts, manifest = discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    columns = pl.scan_parquet(parts).collect_schema().names()\n    features = safe_features(manifest, columns)\n    log(f\"панель: {manifest['rows']:,} строк, частей {len(parts)}, признаков {len(features)}; режим {cfg['mode']}\")\n    cov = coverage(pl, parts, cfg[\"targets\"], cfg)\n    results = {}\n    for t in cfg[\"targets\"]:\n        log(f\"{t}:\")\n        frame = load_target(pl, parts, t, features, cfg)\n        stress = load_stress(pl, parts, t, features, cfg)\n        results[t] = run_target(pl, frame, t, features, cfg, log, stress=stress)\n        del stress\n        del frame\n    contract = backend_contract(results, cov, cfg)\n    status = \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\"\n    payload = {\"schema_version\": SCHEMA_VERSION, \"status\": status, \"mode\": cfg[\"mode\"],\n               \"panel_manifest\": {\"rows\": manifest[\"rows\"], \"taxonomy_sha256\": manifest[\"taxonomy_sha256\"],\n                                  \"parts_sha256\": [p[\"sha256\"] for p in manifest[\"parts\"]],\n                                  \"panel_config\": manifest[\"panel_config\"]},\n               \"protocol\": {\"folds\": FOLDS,\n                            \"excluded_cutoff_dates\": [str(x) for x in EXCLUDED], \"load_end_exclusive\": str(LOAD_END),\n                            \"features\": features, \"models\": MODELS},\n               \"config\": {k: v for k, v in cfg.items() if k not in (\"input_dir\", \"output_dir\")},\n               \"coverage\": cov, \"targets\": results, \"runtime_s\": round(time.time() - t_start, 1)}\n    (out / \"results_targets_v3.json\").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding=\"utf-8\")\n    (out / \"backend_typed_scores_v3.json\").write_text(json.dumps(contract, ensure_ascii=False, indent=2, default=str),\n                                                      encoding=\"utf-8\")\n    (out / \"summary_targets_v3_ru.md\").write_text(summary_markdown(payload, contract), encoding=\"utf-8\")\n    return {\"status\": status, \"runtime_s\": payload[\"runtime_s\"],\n            \"decisions\": {t: r[\"decision\"][\"evidence_level\"] for t, r in results.items()}}\n\n\ndef _f(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\ndef summary_markdown(payload: dict[str, Any], contract: dict[str, Any]) -> str:\n    L = [f\"# Ноутбук 15 — модели на целях панели v3 ({payload['mode']})\", \"\"]\n    if payload[\"mode\"] == \"SMOKE\":\n        L += [\"**SMOKE:** часть каналов, мало деревьев — цифры не сравнимы с FULL.\", \"\"]\n    L += [\"Метки — наблюдаемые события журнала, не подтверждённые отказы. 2021 и январь 2022 исключены; \"\n          \"2025+ не загружается.\", \"\", \"## Итог по целям\", \"\",\n          \"| цель | ТЗ | evidence_level | score_kind | модель | покрытие строк | PR-AUC 2023 | PR-AUC 2024 | unknown 2024 | precision@25/сут (cooldown) 2024, границы |\",\n          \"|---|---|---|---|---|---|---|---|---|---|\"]\n    for c in contract[\"targets\"]:\n        r = payload[\"targets\"][c[\"target\"]]\n        sel = c[\"selected_model\"]\n        pr = [_f(r[\"folds\"][f].get(\"models\", {}).get(sel, {}).get(\"pr_auc\")) if sel else \"—\" for f in (\"fold_2023\", \"fold_2024\")]\n        v = c[\"validation_fold_2024\"] or {}\n        b = v.get(\"budget_25_per_day\") or {}\n        L.append(f\"| `{c['target']}` | {c['tz_item']} | **{c['evidence_level']}** | {c['score_kind']} | {sel or '—'} | \"\n                 f\"{c['coverage_labelled_share']:.3f} | {pr[0]} | {pr[1]} | {_f(v.get('unknown_share_selectable'))} | \"\n                 f\"{_f(b.get('precision_cooldown'))}–{_f(b.get('precision_upper_cooldown'))} |\")\n    for t, r in payload[\"targets\"].items():\n        L += [\"\", f\"## `{t}`\", \"\"]\n        for fname, f in r[\"folds\"].items():\n            L += [f\"**{fname}**: train {f['rows']['train']:,} (1: {f['rows']['train_pos']:,}), \"\n                  f\"validation {f['rows']['validation']:,} (1: {f['rows']['validation_pos']:,})\", \"\"]\n            if \"skipped\" in f:\n                L += [f\"пропущен: {f['skipped']}\", \"\"]\n                continue\n            L += [f\"selectable в валидации: {f['rows']['validation_selectable']:,}, исход неизвестен: \"\n                  f\"{f['rows']['validation_unknown']:,} ({f['unknown_share_selectable']:.1%}). PR-AUC/ROC/Brier/ECE — по известным; \"\n                  \"P — нижняя–верхняя граница по всей selectable когорте, R — по известным положительным.\", \"\",\n                  \"| модель | PR-AUC | ROC-AUC | Brier | ECE | P@10 | R@10 | P@25 cd | R@25 cd | слотов в unknown @25 cd |\",\n                  \"|---|---|---|---|---|---|---|---|---|---|\"]\n            for m, x in f[\"models\"].items():\n                b10, b25 = x[\"budget\"].get(\"10\", {}), x[\"budget\"].get(\"25\", {})\n                L.append(f\"| {m} | {_f(x['pr_auc'])} | {_f(x['roc_auc'])} | {_f(x['brier'], 4)} | {_f(x['ece10'], 4)} | \"\n                         f\"{_f(b10.get('precision'))}–{_f(b10.get('precision_upper'))} | {_f(b10.get('recall'))} | \"\n                         f\"{_f(b25.get('precision_cooldown'))}–{_f(b25.get('precision_upper_cooldown'))} | \"\n                         f\"{_f(b25.get('recall_cooldown'))} | {b25.get('unknown_alerts_cooldown', 0):,} |\")\n            bb = f[\"best_simple_baseline\"]\n            bs = f[f\"bootstrap_G1_lightgbm_vs_{bb}\"]\n            bl = f[f\"bootstrap_L1_logistic_vs_{bb}\"]\n            L += [\"\", f\"G1 − {bb}: ΔPR-AUC {_f(bs['mean'])} [{_f(bs['ci95_low'])}; {_f(bs['ci95_high'])}], повторов {bs['reps']}; \"\n                  f\"L1 − {bb}: {_f(bl['mean'])} [{_f(bl['ci95_low'])}; {_f(bl['ci95_high'])}].\",\n                  \"Топ признаков G1: \" + \", \".join(f\"{k} {v:.2f}\" for k, v in list(f[\"importance_G1\"].items())[:6]), \"\"]\n            b50 = f[\"models\"][\"G1_lightgbm\"][\"budget\"].get(\"50\", {})\n            L += [f\"Top-50/сутки, G1, cooldown: слотов в unknown {b50.get('unknown_alerts_cooldown', 0):,} из {b50.get('alerts_cooldown', 0):,}; \"\n                  f\"recall [{_f(b50.get('recall_low_cooldown'))}; {_f(b50.get('recall_up_cooldown'))}], \"\n                  f\"precision [{_f(b50.get('precision_cooldown'))}; {_f(b50.get('precision_upper_cooldown'))}].\", \"\"]\n            if \"competing_d1\" in f:\n                cd = f[\"competing_d1\"]\n                L += [f\"Onset в D+1 (competing): {_f(cd['share_of_rows'])} строк, {_f(cd['share_of_positives'])} положительных. \"\n                      \"PR-AUC для «первого onset» (onset в D+2 без onset в D+1): \"\n                      + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in cd[\"pr_auc_first_onset\"].items()), \"\"]\n        d = r[\"decision\"]\n        if r.get(\"stress_2021\", {}).get(\"pr_auc\"):\n            st = r[\"stress_2021\"]\n            L += [f\"Stress 2021 (модели fold_2024, в выборе не участвует): строк {st['rows']:,}, prevalence {_f(st['prevalence'], 4)}; PR-AUC \"\n                  + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in st[\"pr_auc\"].items()), \"\"]\n        L += [\"Gates: \" + \", \".join(f\"{k} = {'—' if v is None else ('да' if v else 'нет')}\" for k, v in d[\"gates\"].items()), \"\"]\n    L += [\"\", \"## Как читать\", \"\",\n          \"- `validated_proxy_model` — модель (LightGBM или логистическая регрессия) лучше простых baseline (приор по типу, правило) \"\n          \"в обоих фолдах: нижняя граница 95% CI разницы PR-AUC > 0; при равенстве берётся логистическая регрессия.\",\n          \"- `baseline_only` — модель не доказала преимущества; бэкенд отдаёт скор простого правила/приора.\",\n          \"- `insufficient_events` — мало событий в валидации: бэкенд не выдаёт скор (abstain).\",\n          \"- P@k — precision топ-k строк в сутки среди размеченных строк; `cd` — с cooldown по каналу.\",\n          \"\"]\n    return \"\\n\".join(L)\n\n\n# ---------------------------------------------------------------------------\n# Синтетика для самопроверок и тестов\n# ---------------------------------------------------------------------------\ndef synthetic_panel_v3(out_dir: Path, n_channels: int = 150, seed: int = 3, signal: float = 2.0,\n                       end: date = date(2025, 1, 10), gas_features: bool = False) -> Path:\n    \"\"\"Панель со схемой v3 и встроенным сигналом (для тестов; не реальные данные).\n    gas_features: добавить f_gas_n_7d / f_gas_last и строки above_threshold_at_D для T4 (тесты экспорта 16).\"\"\"\n    import polars as pl\n\n    rng = np.random.default_rng(seed)\n    days = [date(2019, 1, 1) + timedelta(days=i) for i in range(0, (end - date(2019, 1, 1)).days, 3)]\n    n = n_channels * len(days)\n    ch = np.repeat(np.arange(n_channels), len(days))\n    d = np.tile(np.array(days, dtype=\"datetime64[D]\"), n_channels)\n    types = np.array([\"Состояние насоса\", \"КД Дверь\", \"Газовый датчик\", \"Состояние фазы\"])[ch % 4]\n    base = rng.poisson(0.3, n).astype(float)\n    frame = {\n        \"d_channel_key\": np.array([hashlib.sha256(f\"s{c}\".encode()).hexdigest()[:16] for c in range(n_channels)])[ch],\n        \"d_cutoff_date\": d, \"d_label_decision_end\": d + np.timedelta64(4, \"D\"),\n        \"тип_датчика\": types, \"тип_инж_системы\": np.where(ch % 2 == 0, \"ВК\", \"ЭС\"),\n        \"link_state\": rng.choice([\"O\", \"F\", \"A\"], n, p=[0.9, 0.05, 0.05]),\n        \"power_state\": rng.choice([\"O\", \"F\"], n, p=[0.9, 0.1]),\n        \"d_weekday\": (d.astype(\"datetime64[D]\").astype(int) + 3) % 7 + 1,\n        \"d_month\": d.astype(\"datetime64[M]\").astype(int) % 12 + 1,\n        \"f_n_events_7d\": rng.poisson(50, n).astype(float),\n    }\n    for t, spec in TARGET_SPECS.items():\n        for col in spec[\"rule\"]:\n            frame.setdefault(col, base + rng.poisson(0.2, n))\n    noise = rng.normal(0, 1, n)\n    for t, spec in TARGET_SPECS.items():\n        x = frame[spec[\"rule\"][0]]\n        logit = -4.0 + signal * (x - x.mean()) / (x.std() + 1e-9) + 0.3 * noise\n        y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(float)\n        null = rng.random(n) < 0.3\n        frame[t] = np.where(null, np.nan, y)\n        frame[\"reason_\" + t[len(\"target_\"):]] = np.where(null, \"unobserved_window\", np.where(y == 1, \"positive\", \"negative\"))\n    if gas_features:\n        # объект = 8 каналов; ежегодное «молчание» объекта (как ППР): gas_max = null 12 суток в году\n        obj = ch // 8\n        frame[\"d_object_key\"] = np.array([hashlib.sha256(f\"o{o}\".encode()).hexdigest()[:16] for o in range(n_channels // 8 + 1)])[obj]\n        doy = (d.astype(\"datetime64[D]\") - d.astype(\"datetime64[Y]\")).astype(int)\n        silent = np.abs(doy - (40 + 23 * (obj % 12))) < 6\n        frame[\"gas_max\"] = np.where(silent, np.nan, rng.uniform(0, 0.9, n))\n        frame[\"f_gas_cross_n_30d\"] = rng.poisson(0.3, n).astype(float)\n        above = rng.random(n) < 0.1\n        frame[\"f_gas_n_7d\"] = rng.poisson(5, n).astype(float) + 1\n        frame[\"f_gas_last\"] = np.where(above, 1.5, rng.uniform(0, 0.9, n))\n        frame[\"target_t4_gas_cross\"] = np.where(above, np.nan, frame[\"target_t4_gas_cross\"])\n        frame[\"reason_t4_gas_cross\"] = np.where(above, \"above_threshold_at_D\", frame[\"reason_t4_gas_cross\"])\n    panel = pl.DataFrame(frame).with_columns(\n        pl.col(\"d_cutoff_date\").cast(pl.Date), pl.col(\"d_label_decision_end\").cast(pl.Date),\n        *[pl.col(t).fill_nan(None).cast(pl.Int8) for t in TARGET_SPECS])\n    panel_dir = Path(out_dir) / \"event_panel_v3\"\n    panel_dir.mkdir(parents=True, exist_ok=True)\n    part = panel_dir / \"event_panel_v3_part00.parquet\"\n    panel.write_parquet(part)\n    feats = [c for c in panel.columns if c.startswith(\"f_\")] + CATEGORICAL + CALENDAR\n    manifest = {\"schema_version\": SCHEMA_VERSION, \"rows\": panel.height, \"taxonomy_sha256\": \"synthetic\",\n                \"panel_config\": {\"synthetic\": True}, \"feature_columns\": feats, \"target_columns\": list(TARGET_SPECS),\n                \"parts\": [{\"file\": part.name, \"sha256\": file_sha256(part), \"rows\": panel.height}]}\n    (panel_dir / \"panel_manifest_v3.json\").write_text(json.dumps(manifest), encoding=\"utf-8\")\n    return panel_dir\n\n\ndef run_targets_self_tests(work_dir: Path) -> dict[str, bool]:\n    import polars as pl\n\n    work_dir = Path(work_dir)\n    rep: dict[str, bool] = {}\n    y = np.array([0, 1, 0, 1, 0, 0])\n    days = np.array([0, 0, 0, 1, 1, 1])\n    ch = np.array([\"a\", \"b\", \"c\", \"b\", \"a\", \"c\"])\n    s = np.array([0.1, 0.9, 0.2, 0.8, 0.3, 0.1])\n    b = budget_curve(days, ch, y, s, [1], cooldown_days=3)[\"1\"]\n    rep[\"budget_top1_hits_both_days\"] = b[\"precision\"] == 1.0 and b[\"recall\"] == 1.0\n    rep[\"cooldown_blocks_repeat_channel\"] = b[\"alerts_cooldown\"] == 2 and b[\"recall_cooldown\"] == 0.5\n    bu = budget_curve(np.array([0, 0]), np.array([\"a\", \"b\"]), np.array([-1, 1]), np.array([0.9, 0.1]), [1], 3)[\"1\"]\n    rep[\"unknown_slot_counted_as_bounds\"] = (bu[\"precision\"] == 0.0 and bu[\"precision_upper\"] == 1.0 and bu[\"unknown_alerts\"] == 1\n                                             and bu[\"recall_low\"] == 0.0 and bu[\"recall_up\"] == 0.5)\n    try:\n        safe_features({\"feature_columns\": [\"f_x\", \"competing_link_onset_d1\"]}, [\"f_x\", \"competing_link_onset_d1\"])\n        rep[\"future_columns_rejected\"] = False\n    except ValueError:\n        rep[\"future_columns_rejected\"] = True\n    frame = pl.DataFrame({\"d_cutoff_date\": [date(2022, 12, 25), date(2022, 12, 30), date(2023, 6, 29), date(2023, 7, 3)],\n                          \"d_label_decision_end\": [date(2022, 12, 29), date(2023, 1, 3), date(2023, 7, 3), date(2023, 7, 7)]})\n    tr, ca, va = split_fold(pl, frame, FOLDS[0])\n    rep[\"purging_by_label_decision_end\"] = tr.height == 1 and ca.height == 0 and va.height == 1\n    root = work_dir / \"synthetic_in\"\n    synthetic_panel_v3(root, n_channels=60)\n    cfg = make_config_targets(\"SMOKE\", {\"input_dir\": str(root), \"output_dir\": str(work_dir / \"out\"),\n                                        \"smoke_channel_share\": 1, \"targets\": [\"target_t2a_link_onset\"],\n                                        \"bootstrap_reps\": 10})\n    res = run_targets_v3(cfg, log=lambda *a: None)\n    rep[\"synthetic_end_to_end\"] = res[\"status\"] == \"completed_smoke_non_comparable\"\n    text = (work_dir / \"out\" / \"results_targets_v3.json\").read_text(encoding=\"utf-8\")\n    keys = pl.read_parquet(root / \"event_panel_v3\" / \"event_panel_v3_part00.parquet\")[\"d_channel_key\"].unique().to_list()\n    rep[\"no_channel_keys_in_outputs\"] = not any(k in text for k in keys)\n    return rep\n",
"incident_predictor_v1.py": "\"\"\"Predictor модуля прогноза инцидентов по реестру (incident_head v1). Кладётся в bundle как predictor.py.\n\nЕдинственный источник кода агрегации «каналы → объект»: обучение (incident_head_v1.py) импортирует отсюда\naggregate_objects, поэтому признаки при обучении и при выдаче считаются одним и тем же кодом.\n\nВход predict: канальные v3-векторы на конец суток D (как для газового bundle) + ключ объекта и тип датчика.\nВыход: по одной записи на (объект, D, тип инцидента) с калиброванной вероятностью начала инцидента в [D+2; D+3).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nAGG_VERSION = \"object-agg-v1\"\nINCIDENT_TYPES = {\"пожар\": \"fire\", \"подтопление\": \"flood\", \"нсд\": \"access\"}\nTYPE_RU = {v: k for k, v in INCIDENT_TYPES.items()}\n# Группы типов датчиков заданы заранее по названию (без взгляда на метки); первая подходящая группа выигрывает.\nGROUPS = {\n    \"fire\": r\"газ|дым|тепл|температ|пожар|пламен|извещат\",\n    \"flood\": r\"затоп|уров|насос|вод|дренаж|приямк|залив\",\n    \"access\": r\"движ|двер|люк|охран|кд|доступ|проник\",\n}\nGROUP_RU = {\"fire\": \"пожарные датчики\", \"flood\": \"датчики воды и насосы\", \"access\": \"датчики доступа\", \"other\": \"прочие датчики\"}\nCHANNEL_SUM = [\"n_events\", \"n_alarm\", \"n_link_fault\", \"n_power_off\", \"f_n_events_7d\", \"f_n_alarm_7d\", \"f_n_alarm_30d\",\n               \"f_n_event_alarm_7d\", \"f_n_flood_7d\", \"f_n_flood_30d\", \"f_n_link_fault_7d\", \"f_n_power_off_7d\",\n               \"f_link_onset_strict_7d\", \"f_power_onset_strict_7d\", \"f_n_tech_like_7d\", \"f_gas_ge_thr_n_7d\",\n               \"f_gas_cross_n_30d\"]\nGROUP_COLS = [\"n_alarm\", \"n_events\", \"f_n_alarm_7d\", \"f_n_alarm_30d\", \"f_n_events_7d\"]\nCOL_RU = {\"n_events\": \"события за сутки D\", \"n_alarm\": \"тревоги за сутки D\", \"n_link_fault\": \"потери связи за сутки D\",\n          \"n_power_off\": \"обесточивания за сутки D\", \"f_n_events_7d\": \"события за 7 сут\", \"f_n_alarm_7d\": \"тревоги за 7 сут\",\n          \"f_n_alarm_30d\": \"тревоги за 30 сут\", \"f_n_event_alarm_7d\": \"тревожные события за 7 сут\",\n          \"f_n_flood_7d\": \"сигналы затопления за 7 сут\", \"f_n_flood_30d\": \"сигналы затопления за 30 сут\",\n          \"f_n_link_fault_7d\": \"потери связи за 7 сут\", \"f_n_power_off_7d\": \"обесточивания за 7 сут\",\n          \"f_link_onset_strict_7d\": \"новые потери связи за 7 сут\", \"f_power_onset_strict_7d\": \"новые обесточивания за 7 сут\",\n          \"f_n_tech_like_7d\": \"технические значения за 7 сут\", \"f_gas_ge_thr_n_7d\": \"газ ≥ 1 % за 7 сут\",\n          \"f_gas_cross_n_30d\": \"пересечения газа 1 % за 30 сут\"}\n# Правило-baseline для каждого типа: профильные признаки на конец D (сумма)\nRULE_COLS = {\"fire\": [\"grp_fire__f_n_alarm_7d\", \"obj__f_gas_ge_thr_n_7d\"],\n             \"flood\": [\"grp_flood__f_n_alarm_7d\", \"obj__f_n_flood_7d\"],\n             \"access\": [\"grp_access__f_n_alarm_7d\"]}\nKEYS = [\"object_ref\", \"d_cutoff_date\"]\n\n\nclass ContractError(ValueError):\n    pass\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef group_expr(pl: Any, type_col: str = \"тип_датчика\") -> Any:\n    t = pl.col(type_col).cast(pl.String).fill_null(\"\").str.to_lowercase()\n    expr = pl.lit(\"other\")\n    for g in reversed(list(GROUPS)):\n        expr = pl.when(t.str.contains(GROUPS[g])).then(pl.lit(g)).otherwise(expr)\n    return expr.alias(\"_grp\")\n\n\ndef aggregate_objects(pl: Any, channels: Any, object_col: str = \"d_object_key\") -> Any:\n    \"\"\"Каналы (LazyFrame/DataFrame, строка = канал × D) → объект × D. Колонки, которых нет во входе, пропускаются;\n    возвращаются также служебные колонки следа _fp_<тип> (профильные тревоги самих суток; не признаки).\"\"\"\n    eager = not isinstance(channels, pl.LazyFrame)\n    lf = channels.lazy() if eager else channels\n    cols = set(lf.collect_schema().names())\n    lf = lf.filter(pl.col(object_col).is_not_null()).with_columns(group_expr(pl))\n    aggs = [pl.len().alias(\"obj__n_channels\")]\n    aggs += [pl.col(c).cast(pl.Float64).sum().alias(f\"obj__{c}\") for c in CHANNEL_SUM if c in cols]\n    for g in list(GROUPS) + [\"other\"]:\n        m = pl.col(\"_grp\") == g\n        aggs.append(m.sum().cast(pl.Float64).alias(f\"grp_{g}__n_channels\"))\n        aggs += [pl.col(c).cast(pl.Float64).filter(m).sum().alias(f\"grp_{g}__{c}\") for c in GROUP_COLS if c in cols]\n    if \"link_state\" in cols:\n        aggs.append((pl.col(\"link_state\").cast(pl.String) != \"O\").mean().cast(pl.Float64).alias(\"obj__share_link_not_ok\"))\n    if \"power_state\" in cols:\n        aggs.append((pl.col(\"power_state\").cast(pl.String) == \"F\").mean().cast(pl.Float64).alias(\"obj__share_power_off\"))\n    if \"gas_max\" in cols:\n        g = pl.col(\"gas_max\").cast(pl.Float64).fill_nan(None)\n        aggs += [g.max().alias(\"obj__gas_max\"), (g >= 1.0).sum().cast(pl.Float64).alias(\"_gas_ge1_channels\")]\n    if \"f_gas_max_7d\" in cols:\n        aggs.append(pl.col(\"f_gas_max_7d\").cast(pl.Float64).fill_nan(None).max().alias(\"obj__f_gas_max_7d\"))\n    out = lf.group_by(pl.col(object_col).alias(\"object_ref\"), \"d_cutoff_date\").agg(aggs)\n    names = out.collect_schema().names()\n    fire_fp = pl.col(\"grp_fire__n_alarm\").fill_null(0) if \"grp_fire__n_alarm\" in names else pl.lit(0.0)\n    if \"_gas_ge1_channels\" in names:\n        fire_fp = fire_fp + pl.col(\"_gas_ge1_channels\").fill_null(0)\n    fp = {\"fire\": fire_fp,\n          \"flood\": pl.col(\"grp_flood__n_alarm\").fill_null(0) if \"grp_flood__n_alarm\" in names else pl.lit(0.0),\n          \"access\": pl.col(\"grp_access__n_alarm\").fill_null(0) if \"grp_access__n_alarm\" in names else pl.lit(0.0)}\n    out = out.with_columns([e.cast(pl.Float64).alias(f\"_fp_{k}\") for k, e in fp.items()]\n                           + [pl.col(\"d_cutoff_date\").dt.month().cast(pl.Float64).alias(\"d_month\")])\n    drop = [c for c in [\"_gas_ge1_channels\"] if c in names]\n    out = out.drop(drop)\n    return out.collect() if eager else out\n\n\ndef feature_columns(columns: list[str]) -> list[str]:\n    return [c for c in columns if c not in KEYS and not c.startswith(\"_\")]\n\n\ndef humanize(feature: str) -> str:\n    if feature == \"d_month\":\n        return \"месяц (сезонность)\"\n    m = re.match(r\"^(obj|grp_(\\w+?))__(.+)$\", feature)\n    if not m:\n        return feature\n    scope = \"объект\" if m.group(1) == \"obj\" else GROUP_RU.get(m.group(2), m.group(2))\n    col = m.group(3)\n    names = {\"n_channels\": \"число каналов\", \"share_link_not_ok\": \"доля каналов без связи\",\n             \"share_power_off\": \"доля обесточенных каналов\", \"gas_max\": \"максимум газа за сутки D\",\n             \"f_gas_max_7d\": \"максимум газа за 7 сут\"}\n    return f\"{scope}: {names.get(col, COL_RU.get(col, col))}\"\n\n\ndef rule_raw(frame: Any, kind: str) -> np.ndarray:\n    s = np.zeros(frame.height)\n    for c in RULE_COLS[kind]:\n        if c in frame.columns:\n            s = s + np.nan_to_num(frame[c].cast(float).to_numpy(), nan=0.0)\n    return s\n\n\ndef _platt(raw: np.ndarray, p: dict[str, Any] | None, kind: str) -> np.ndarray:\n    if p is None:\n        return np.full(len(raw), np.nan)\n    if kind == \"probability\":\n        q = np.clip(raw, 1e-6, 1 - 1e-6)\n        z = np.log(q / (1 - q))\n    else:\n        z = np.log1p(np.maximum(raw, 0))\n    return 1 / (1 + np.exp(-(p[\"coef\"] * z + p[\"intercept\"])))\n\n\n# ---------------------------------------------------------------------------\n# Bundle\n# ---------------------------------------------------------------------------\ndef load_bundle(bundle_dir: str | Path) -> dict[str, Any]:\n    import lightgbm as lgb\n\n    bdir = Path(bundle_dir)\n    sums = json.loads((bdir / \"SHA256SUMS.json\").read_text(encoding=\"utf-8\"))\n    for name, sha in sums.items():\n        if _sha256_bytes((bdir / name).read_bytes()) != sha:\n            raise ContractError(f\"SHA-256 {name} не совпадает с SHA256SUMS.json\")\n    spec = json.loads((bdir / \"bundle.json\").read_text(encoding=\"utf-8\"))\n    models = {}\n    for kind, t in spec[\"types\"].items():\n        if t.get(\"model_file\"):\n            booster = lgb.Booster(model_str=(bdir / t[\"model_file\"]).read_text(encoding=\"utf-8\"))\n            if list(booster.feature_name()) != [f.replace(\" \", \"_\") for f in spec[\"features\"]]:\n                raise ContractError(f\"признаки модели {kind} не совпадают с контрактом\")\n            models[kind] = booster\n    return {\"spec\": spec, \"models\": models, \"dir\": str(bdir)}\n\n\ndef _as_frame(pl: Any, records: Any) -> Any:\n    if hasattr(records, \"lazy\"):\n        return records\n    if isinstance(records, list):\n        return pl.DataFrame(records, infer_schema_length=None)\n    return pl.from_pandas(records)\n\n\ndef predict(channel_records: Any, bundle: dict[str, Any], as_of_date: str | date, request_id: str | None = None,\n            object_col: str = \"ид_объект\", recent_incidents: list[dict[str, Any]] | None = None,\n            top_factors: int = 3) -> list[dict[str, Any]]:\n    \"\"\"channel_records: канальные v3-векторы на конец as_of_date (polars/pandas/list of dict) с колонками object_col\n    и тип_датчика. recent_incidents: [{object_col: …, \"тип_инцидента\": …, \"начало\": \"YYYY-MM-DD\"}] — начатые в [D−6; D].\"\"\"\n    import polars as pl\n\n    spec = bundle[\"spec\"]\n    if not isinstance(as_of_date, date):\n        if not re.fullmatch(r\"\\d{4}-\\d{2}-\\d{2}\", str(as_of_date or \"\")):\n            raise ContractError(\"as_of_date обязательна, формат YYYY-MM-DD\")\n        as_of_date = date.fromisoformat(str(as_of_date))\n    ch = _as_frame(pl, channel_records)\n    if object_col not in ch.columns or \"тип_датчика\" not in ch.columns:\n        raise ContractError(f\"нужны колонки {object_col} и тип_датчика\")\n    ch = ch.with_columns(pl.lit(as_of_date).alias(\"d_cutoff_date\"))\n    obj = aggregate_objects(pl, ch, object_col=object_col)\n    feats = spec[\"features\"]\n    for f in feats:\n        if f not in obj.columns:\n            obj = obj.with_columns(pl.lit(None, dtype=pl.Float64).alias(f))\n    x = obj.select([pl.col(f).cast(pl.Float64) for f in feats]).to_numpy()\n    ongoing = {(str(r[object_col]), INCIDENT_TYPES.get(r[\"тип_инцидента\"], r[\"тип_инцидента\"]))\n               for r in (recent_incidents or [])\n               if as_of_date - timedelta(days=6) <= date.fromisoformat(str(r[\"начало\"])[:10]) <= as_of_date}\n    win_s, win_e = as_of_date + timedelta(days=2), as_of_date + timedelta(days=3)\n    covered = spec[\"coverage\"].get(\"objects\", \"all\")\n    covered = None if covered == \"all\" else {str(o) for o in covered}\n    out: list[dict[str, Any]] = []\n    refs = [str(r) for r in obj[\"object_ref\"].to_list()]\n    for kind, t in spec[\"types\"].items():\n        status = t[\"decision_status\"]\n        score = np.full(len(refs), np.nan)\n        contrib = None\n        if status in (\"experimental_shadow\", \"demo_stub\") and kind in bundle[\"models\"]:\n            b = bundle[\"models\"][kind]\n            score = _platt(b.predict(x), t[\"platt\"], \"probability\")\n            if top_factors:\n                contrib = b.predict(x, pred_contrib=True)[:, :-1]\n        elif status == \"baseline_only\":\n            score = _platt(rule_raw(obj, kind), t[\"rule_platt\"], \"count\")\n        for i, ref in enumerate(refs):\n            reason = None\n            if status == \"insufficient_labels\":\n                reason = \"INSUFFICIENT_LABELS\"\n            elif covered is not None and ref not in covered:\n                reason = \"OBJECT_NOT_COVERED\"\n            elif (ref, kind) in ongoing:\n                reason = \"ONGOING_INCIDENT\"\n            s = None if reason or np.isnan(score[i]) else float(score[i])\n            factors = []\n            if s is not None and contrib is not None:\n                order = np.argsort(-contrib[i])[:top_factors]\n                factors = [{\"feature\": humanize(feats[j]), \"direction\": \"up\"} for j in order if contrib[i, j] > 0]\n            elif s is not None and status == \"baseline_only\":\n                factors = [{\"feature\": humanize(c), \"direction\": \"up\"} for c in RULE_COLS[kind]]\n            out.append({\"request_id\": request_id, object_col: ref, \"as_of_date\": as_of_date.isoformat(),\n                        \"incident_type\": TYPE_RU[kind], \"window_start\": win_s.isoformat(),\n                        \"window_end_exclusive\": win_e.isoformat(), \"score\": s,\n                        \"score_kind\": \"calibrated_probability\" if s is not None else None,\n                        \"evidence_level\": spec[\"evidence_level\"], \"label_source\": spec[\"label_source\"],\n                        \"decision_status\": status, \"operational_ready\": False, \"abstain_reason\": reason,\n                        \"top_factors\": factors, \"model_version\": spec[\"model_version\"],\n                        \"feature_contract_version\": spec[\"feature_contract_version\"]})\n    return out\n",
"incident_head_v1.py": "\"\"\"Модуль прогноза инцидентов по реестру (incident_head v1, ноутбук 18).\n\nКонтракт: docs/INCIDENT_REGISTER_CONTRACT_RU.md.\n- Вход: панель v3 (выход 14) + реестр подтверждённых инцидентов с файлом покрытия. Пока реального реестра нет,\n  используется ЗАГЛУШКА (label_source = \"stub\", evidence_level = E4); реальный реестр подключается тем же кодом.\n- Цель на (объект, D) для типа k: 1 — подтверждённый инцидент типа k начался в [D+2; D+3); 0 — в покрытии и нет;\n  null — вне покрытия или инцидент того же типа начался в [D−6; D].\n- Признаки: агрегация канальных v3-векторов до объекта (incident_predictor_v1.aggregate_objects — тот же код в bundle).\n- Модель: LightGBM + Platt, разбиение по времени; заранее заданные пороги допуска; baseline — prevalence и правило\n  «профильные тревоги за 7 суток»; кривая обучения по числу положительных.\n\nЗаглушка: инцидент типа k разыгрывается по реальным суткам X объекта, logit = a + β·z(log1p(след_k(X))), где след —\nпрофильные тревоги объекта в сами сутки X; плюс доля инцидентов без следа. Модель видит только данные ≤ D = X − 2,\nпоэтому измеряется предсказуемость будущего всплеска профильной активности, а не рецепт генератора.\nβ: 0 (контроль — качество обязано быть на уровне случайного), 1 (слабая), 2 (средняя, основная для bundle), 3 (сильная:\nнастоящий пожар или затопление почти всегда даёт срабатывания профильных датчиков). Всё задано до запуска на реальных данных.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nimport incident_predictor_v1 as ip\nimport target_models_v3_runtime as rt\n\nMODULE_DIR = Path(__file__).resolve().parent\nSCENARIOS = {\"control\": 0.0, \"weak\": 1.0, \"medium\": 2.0, \"strong\": 3.0}\nMAIN_SCENARIO = \"medium\"\nPANEL_START = date(2022, 1, 31)       # как в 15/16: 2021 и прогрев окон исключены\nREGISTER_COLUMNS = [\"ид_инцидента\", \"тип_инцидента\", \"начало\", \"подтверждён\"]\nSTUB_GENERATOR_VERSION = \"stub-trace-v1\"\n\n\ndef make_config_18(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    cfg = rt.make_config_targets(mode)\n    cfg.update({\n        \"scenarios\": dict(SCENARIOS), \"main_scenario\": MAIN_SCENARIO,\n        \"rate_per_object_year\": 1.0, \"no_trace_share\": 0.3, \"stub_seed\": 18,\n        \"stub_period\": (PANEL_START, date(2026, 1, 1)),\n        \"train_end\": date(2025, 1, 1), \"cal_end\": date(2025, 7, 1), \"test_end\": date(2026, 1, 1),\n        \"min_train_positives\": 30, \"min_calibration_positives\": 30, \"min_test_positives\": 10,\n        \"budget_objects_per_day\": 10, \"cooldown_days\": 3,\n        \"learning_curve_sizes\": [25, 50, 100, 200],\n        \"smoke_object_share\": 4,\n        \"register_path\": None, \"coverage_path\": None,\n    })\n    cfg.update(overrides or {})\n    return cfg\n\n\ndef _sha256_file(path: Path) -> str:\n    return rt.file_sha256(Path(path))\n\n\ndef _jd(o: Any) -> Any:\n    if isinstance(o, np.generic):\n        return o.item()\n    if isinstance(o, (date, datetime)):\n        return o.isoformat()\n    raise TypeError(type(o).__name__)\n\n\ndef _dump(obj: Any) -> str:\n    return json.dumps(obj, ensure_ascii=False, indent=2, default=_jd)\n\n\n# ---------------------------------------------------------------------------\n# Объектная панель\n# ---------------------------------------------------------------------------\ndef build_object_panel(pl: Any, parts: list[Path], cfg: dict[str, Any]) -> Any:\n    lf = pl.scan_parquet(parts)\n    cols = lf.collect_schema().names()\n    need = set(ip.CHANNEL_SUM + ip.GROUP_COLS + [\"link_state\", \"power_state\", \"gas_max\", \"f_gas_max_7d\",\n                                                \"тип_датчика\", \"d_object_key\", \"d_cutoff_date\"])\n    lf = lf.select([c for c in cols if c in need]).filter(\n        (pl.col(\"d_cutoff_date\") >= cfg[\"stub_period\"][0]) & (pl.col(\"d_cutoff_date\") < cfg[\"test_end\"] + timedelta(days=3)))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_object_key\").hash(seed=7) % cfg[\"smoke_object_share\"] == 0)\n    return ip.aggregate_objects(pl, lf, object_col=\"d_object_key\").collect().sort(\"object_ref\", \"d_cutoff_date\")\n\n\n# ---------------------------------------------------------------------------\n# Реестр: заглушка и загрузка\n# ---------------------------------------------------------------------------\ndef _solve_intercept(z: np.ndarray, beta: float, target: float) -> float:\n    lo, hi = -40.0, 10.0\n    for _ in range(80):\n        mid = (lo + hi) / 2\n        if np.mean(1 / (1 + np.exp(-(mid + beta * z)))) > target:\n            hi = mid\n        else:\n            lo = mid\n    return (lo + hi) / 2\n\n\ndef make_stub_register(pl: Any, obj: Any, cfg: dict[str, Any], beta: float, seed: int) -> tuple[Any, dict[str, Any]]:\n    \"\"\"Заглушка реестра на реальных сутках объектов. Возвращает (реестр, покрытие).\"\"\"\n    start, end = cfg[\"stub_period\"]\n    fr = obj.filter((pl.col(\"d_cutoff_date\") >= start) & (pl.col(\"d_cutoff_date\") < end))\n    rng = np.random.default_rng(seed)\n    r = cfg[\"rate_per_object_year\"] / 365.0\n    s = cfg[\"no_trace_share\"]\n    rows: dict[str, list[Any]] = {\"ид_инцидента\": [], \"d_object_key\": [], \"тип_инцидента\": [], \"начало\": [],\n                                  \"окончание\": [], \"подтверждён\": [], \"источник\": []}\n    for kind in ip.GROUPS:\n        x = np.log1p(fr[f\"_fp_{kind}\"].fill_null(0).to_numpy().astype(float))\n        z = (x - x.mean()) / x.std() if x.std() > 0 else np.zeros_like(x)\n        a = _solve_intercept(z, beta, r)\n        p = (1 - s) / (1 + np.exp(-(a + beta * z))) + s * r\n        hit = np.flatnonzero(rng.random(len(p)) < p)\n        refs, days = fr[\"object_ref\"].to_numpy()[hit], fr[\"d_cutoff_date\"].to_numpy()[hit]\n        hours = rng.integers(0, 24, len(hit))\n        for i, (o, d, h) in enumerate(zip(refs, days, hours)):\n            t0 = datetime.combine(date.fromisoformat(str(d)[:10]), datetime.min.time()) + timedelta(hours=int(h))\n            rows[\"ид_инцидента\"].append(f\"stub-{kind}-{i:06d}\")\n            rows[\"d_object_key\"].append(str(o))\n            rows[\"тип_инцидента\"].append(ip.TYPE_RU[kind])\n            rows[\"начало\"].append(t0.isoformat())\n            rows[\"окончание\"].append((t0 + timedelta(hours=6)).isoformat())\n            rows[\"подтверждён\"].append(True)\n            rows[\"источник\"].append(\"stub\")\n    reg = pl.DataFrame(rows, schema={k: (pl.Boolean if k == \"подтверждён\" else pl.String) for k in rows})\n    coverage = {\"period_start\": start.isoformat(), \"period_end_exclusive\": end.isoformat(), \"objects\": \"all\",\n                \"incident_types\": list(ip.INCIDENT_TYPES), \"label_source\": \"stub\",\n                \"stub\": {\"generator_version\": STUB_GENERATOR_VERSION, \"beta\": beta,\n                         \"rate_per_object_year\": cfg[\"rate_per_object_year\"],\n                         \"no_trace_share\": s, \"seed\": seed}}\n    return reg, coverage\n\n\ndef validate_coverage(coverage: dict[str, Any]) -> dict[str, Any]:\n    for k in (\"period_start\", \"period_end_exclusive\", \"objects\", \"label_source\"):\n        if k not in coverage:\n            raise ValueError(f\"в покрытии нет поля {k}\")\n    if coverage[\"label_source\"] not in (\"register\", \"stub\"):\n        raise ValueError(\"label_source должен быть register или stub\")\n    return coverage\n\n\ndef _object_key(pl: Any, reg: Any) -> Any:\n    if \"d_object_key\" in reg.columns:\n        return reg.with_columns(pl.col(\"d_object_key\").cast(pl.String).alias(\"object_ref\"))\n    if \"ид_объект\" not in reg.columns:\n        raise ValueError(\"в реестре нет ид_объект\")\n    sys.path.insert(0, str(MODULE_DIR))\n    import event_panel_v3 as ep  # псевдонимы как в панели 14\n\n    return reg.with_columns(pl.col(\"ид_объект\").cast(pl.String).map_elements(ep.pseudo_key, return_dtype=pl.String)\n                            .alias(\"object_ref\"))\n\n\ndef load_register(pl: Any, reg: Any, coverage: dict[str, Any]) -> tuple[Any, dict[str, Any]]:\n    \"\"\"Реестр → события (object_ref, day, kind); только подтверждённые, только три типа ТЗ.\"\"\"\n    validate_coverage(coverage)\n    miss = [c for c in REGISTER_COLUMNS if c not in reg.columns]\n    if miss:\n        raise ValueError(f\"в реестре нет колонок {miss}\")\n    reg = _object_key(pl, reg)\n    ev = (reg.filter(pl.col(\"подтверждён\").cast(pl.Boolean) & pl.col(\"тип_инцидента\").is_in(list(ip.INCIDENT_TYPES)))\n          .select(\"object_ref\", pl.col(\"начало\").cast(pl.String).str.slice(0, 10).str.to_date().alias(\"day\"),\n                  pl.col(\"тип_инцидента\").replace_strict(ip.INCIDENT_TYPES).alias(\"kind\"))\n          .unique())\n    stats = {\"rows\": reg.height, \"confirmed_in_scope\": ev.height,\n             \"by_type\": {k: int((ev[\"kind\"] == k).sum()) for k in ip.GROUPS}}\n    return ev, stats\n\n\n# ---------------------------------------------------------------------------\n# Цели\n# ---------------------------------------------------------------------------\ndef _flag(pl: Any, frame: Any, days: Any, offset: int) -> np.ndarray:\n    key = frame.select(pl.int_range(0, pl.len(), dtype=pl.UInt32).alias(\"_row\"), \"object_ref\",\n                       (pl.col(\"d_cutoff_date\") + pl.duration(days=offset)).alias(\"day\"))\n    hits = key.join(days.select(\"object_ref\", \"day\").unique(), on=[\"object_ref\", \"day\"], how=\"semi\")[\"_row\"].to_numpy()\n    out = np.zeros(frame.height, dtype=bool)\n    out[hits] = True\n    return out\n\n\ndef build_targets(pl: Any, obj: Any, events: Any, coverage: dict[str, Any]) -> Any:\n    start = date.fromisoformat(coverage[\"period_start\"])\n    end = date.fromisoformat(coverage[\"period_end_exclusive\"])\n    d2 = obj[\"d_cutoff_date\"].to_numpy() + np.timedelta64(2, \"D\")\n    in_cov = (d2 >= np.datetime64(start)) & (d2 < np.datetime64(end))\n    if coverage[\"objects\"] != \"all\":\n        tmp = pl.DataFrame({\"object_ref\": [str(o) for o in coverage[\"objects\"]]})\n        if coverage.get(\"objects_are_raw_ids\", True) and coverage[\"label_source\"] == \"register\":\n            tmp = _object_key(pl, tmp.rename({\"object_ref\": \"ид_объект\"}))\n        in_cov &= obj[\"object_ref\"].is_in(tmp[\"object_ref\"]).to_numpy()\n    cols = {}\n    for kind in ip.GROUPS:\n        ev = events.filter(pl.col(\"kind\") == kind)\n        pos = _flag(pl, obj, ev, 2)\n        ongoing = np.zeros(obj.height, dtype=bool)\n        for lag in range(0, 7):                              # начало в [D−6; D] ⇔ day + lag == D\n            ongoing |= _flag(pl, obj, ev, -lag)\n        competing = _flag(pl, obj, ev, 1) & ~pos             # начало того же типа в D+1: событие до окна D+2 (GPT 51)\n        y = np.where(pos, 1.0, 0.0)\n        y = np.where(~in_cov | ongoing | competing, np.nan, y)\n        cols[f\"target_{kind}\"] = y\n        cols[f\"reason_{kind}\"] = np.where(~in_cov, \"outside_coverage\", np.where(ongoing, \"ongoing_incident\", np.where(\n            competing, \"competing_event_d1\", np.where(pos, \"positive\", \"negative\"))))\n    return obj.with_columns([pl.Series(k, v, nan_to_null=True) if k.startswith(\"target_\") else pl.Series(k, v)\n                             for k, v in cols.items()])\n\n\ndef split(pl: Any, frame: Any, cfg: dict[str, Any]) -> tuple[Any, Any, Any]:\n    \"\"\"Разбиение по времени; окно исхода D+2 не пересекает границу периода.\"\"\"\n    d2 = pl.col(\"d_cutoff_date\") + pl.duration(days=2)\n    lo = cfg[\"stub_period\"][0] if cfg.get(\"register_path\") is None else frame[\"d_cutoff_date\"].min()\n    tr = frame.filter((pl.col(\"d_cutoff_date\") >= lo) & (d2 < cfg[\"train_end\"]))\n    ca = frame.filter((pl.col(\"d_cutoff_date\") >= cfg[\"train_end\"]) & (d2 < cfg[\"cal_end\"]))\n    te = frame.filter((pl.col(\"d_cutoff_date\") >= cfg[\"cal_end\"]) & (d2 < cfg[\"test_end\"]))\n    return tr, ca, te\n\n\n# ---------------------------------------------------------------------------\n# Обучение одного типа\n# ---------------------------------------------------------------------------\ndef _lr_params(m: Any) -> dict[str, float] | None:\n    return None if m is None else {\"coef\": float(m.coef_[0][0]), \"intercept\": float(m.intercept_[0])}\n\n\ndef fit_type(pl: Any, tr: Any, ca: Any, te: Any, kind: str, feats: list[str], cfg: dict[str, Any],\n             n_pos: int | None = None, seed: int = 0, full: bool = True) -> dict[str, Any]:\n    from sklearn.metrics import average_precision_score\n\n    tgt = f\"target_{kind}\"\n    known = pl.col(tgt).is_not_null()\n    tr, ca, te = tr.filter(known), ca.filter(known), te.filter(known)\n    counts = {\"train\": tr.height, \"train_pos\": int(tr[tgt].sum() or 0), \"calibration\": ca.height,\n              \"calibration_pos\": int(ca[tgt].sum() or 0), \"test\": te.height, \"test_pos\": int(te[tgt].sum() or 0)}\n    res: dict[str, Any] = {\"counts\": counts}\n    if counts[\"train_pos\"] < cfg[\"min_train_positives\"] or counts[\"calibration_pos\"] < cfg[\"min_calibration_positives\"]:\n        res[\"gate\"] = \"insufficient_labels\"\n        return res\n    if n_pos is not None:\n        if n_pos >= counts[\"train_pos\"]:\n            return {\"counts\": counts, \"gate\": \"skipped_size\"}\n        pos_idx = np.flatnonzero(tr[tgt].to_numpy() == 1)\n        keep = np.random.default_rng(seed).choice(pos_idx, n_pos, replace=False)\n        mask = tr[tgt].to_numpy() == 0\n        mask[keep] = True\n        tr = tr.filter(pl.Series(mask))\n    tr_s = rt.sample_train(pl, tr, tgt, cfg)\n    model = rt.LightGBM().fit(tr_s, tgt, feats, cfg)\n    y_ca, y_te = ca[tgt].to_numpy().astype(int), te[tgt].to_numpy().astype(int)\n    p_model = rt.platt(model.score(ca), y_ca, True)\n    p_rule = rt.platt(ip.rule_raw(ca, kind), y_ca, False)\n    raw_te = model.score(te)\n    s_model = rt.apply_platt(p_model, raw_te, True, float(y_ca.mean()))\n    s_rule = rt.apply_platt(p_rule, ip.rule_raw(te, kind), False, float(y_ca.mean()))\n    prev = float(y_te.mean()) if len(y_te) else None\n    ok = counts[\"test_pos\"] >= cfg[\"min_test_positives\"] and 0 < y_te.sum() < len(y_te)\n    m: dict[str, Any] = {\"prevalence\": prev}\n    if ok:\n        m.update({\"pr_auc_model\": float(average_precision_score(y_te, raw_te)),\n                  \"pr_auc_rule\": float(average_precision_score(y_te, s_rule))})\n        m[\"lift_model\"], m[\"lift_rule\"] = m[\"pr_auc_model\"] / prev, m[\"pr_auc_rule\"] / prev\n    res.update({\"n_pos_used\": n_pos, \"test_metrics\": m})\n    if not full:\n        return res\n    days = np.array([(d - date(2019, 1, 1)).days for d in te[\"d_cutoff_date\"].to_list()])\n    refs = te[\"object_ref\"].to_numpy()\n    if ok:\n        m.update({\"brier_model\": float(np.mean((s_model - y_te) ** 2)), \"ece10_model\": rt.ece(y_te, s_model),\n                  \"bootstrap_model_minus_rule\": rt.paired_bootstrap(y_te, raw_te, s_rule, days // 7,\n                                                                   cfg[\"bootstrap_reps\"], cfg[\"random_seed\"])})\n        k = cfg[\"budget_objects_per_day\"]\n        for name, sc in ((\"model\", s_model), (\"rule\", s_rule)):\n            b = rt.budget_curve(days, refs, y_te, sc, [k], cfg[\"cooldown_days\"])[str(k)]\n            m[f\"top{k}_{name}\"] = {x: b[x] for x in (\"alerts_cooldown\", \"precision_cooldown\", \"recall_cooldown\")}\n    ci = (m.get(\"bootstrap_model_minus_rule\") or {}).get(\"ci95_low\")\n    res[\"gate\"] = \"passed\" if (ci is not None and ci > 0) else \"baseline_only\"\n    res.update({\"model\": model, \"platt\": _lr_params(p_model), \"rule_platt\": _lr_params(p_rule),\n                \"importance\": model.importance(8)})\n    return res\n\n\ndef learning_curve(pl: Any, tr: Any, ca: Any, te: Any, kind: str, feats: list[str], cfg: dict[str, Any]) -> list[dict[str, Any]]:\n    out = []\n    for n in cfg[\"learning_curve_sizes\"]:\n        r = fit_type(pl, tr, ca, te, kind, feats, cfg, n_pos=n, seed=cfg[\"random_seed\"], full=False)\n        if r.get(\"gate\") in (\"insufficient_labels\", \"skipped_size\"):\n            continue\n        out.append({\"n_train_positives\": n, **{k: r[\"test_metrics\"].get(k) for k in (\"pr_auc_model\", \"lift_model\", \"prevalence\")}})\n    return out\n\n\ndef decision_status(gate: str, label_source: str) -> str:\n    if gate == \"insufficient_labels\":\n        return \"insufficient_labels\"\n    if gate == \"baseline_only\":\n        return \"baseline_only\"\n    return \"demo_stub\" if label_source == \"stub\" else \"experimental_shadow\"\n\n\n# ---------------------------------------------------------------------------\n# Bundle\n# ---------------------------------------------------------------------------\ndef export_bundle(out_dir: Path, fits: dict[str, dict[str, Any]], feats: list[str], coverage: dict[str, Any],\n                  register_sha: str, manifest: dict[str, Any], manifest_path: Path, cfg: dict[str, Any]) -> Path:\n    bdir = Path(out_dir) / \"incident_head_bundle\"\n    if bdir.exists():\n        shutil.rmtree(bdir)\n    bdir.mkdir(parents=True)\n    label_source = coverage[\"label_source\"]\n    types, model_shas = {}, []\n    for kind, f in fits.items():\n        status = decision_status(f[\"gate\"], label_source)\n        t: dict[str, Any] = {\"decision_status\": status, \"counts\": f[\"counts\"], \"test_metrics\": _public(f.get(\"test_metrics\")),\n                             \"platt\": f.get(\"platt\"), \"rule_platt\": f.get(\"rule_platt\"), \"rule_features\": ip.RULE_COLS[kind],\n                             \"model_file\": None}\n        if \"model\" in f and status in (\"demo_stub\", \"experimental_shadow\"):\n            text = f[\"model\"].model.booster_.model_to_string()\n            (bdir / f\"model_{kind}.txt\").write_text(text, encoding=\"utf-8\")\n            t[\"model_file\"] = f\"model_{kind}.txt\"\n            model_shas.append(hashlib.sha256(text.encode(\"utf-8\")).hexdigest())\n        types[kind] = t\n    tag = hashlib.sha256(\"\".join(model_shas).encode()).hexdigest()[:12] if model_shas else \"rules\"\n    spec = {\n        \"module\": \"incident_head\", \"agg_version\": ip.AGG_VERSION,\n        \"model_version\": f\"incident-head-v1-{label_source}-{tag}\",\n        \"feature_contract_version\": f\"event-panel-v{manifest.get('schema_version')}+{ip.AGG_VERSION}\",\n        \"label_source\": label_source, \"evidence_level\": \"E0\" if label_source == \"register\" else \"E4\",\n        \"operational_ready\": False, \"window\": \"[D+2; D+3)\", \"features\": feats, \"types\": types,\n        \"coverage\": coverage, \"register_sha256\": register_sha,\n        \"generator_version\": (coverage.get(\"stub\") or {}).get(\"generator_version\"),\n        \"score_meaning\": (\"demo-score на искусственных метках, НЕ вероятность подтверждённого инцидента\"\n                          if label_source == \"stub\" else \"вероятность начала подтверждённого инцидента в окне\"),\n        \"split\": {\"train_end\": cfg[\"train_end\"], \"cal_end\": cfg[\"cal_end\"], \"test_end\": cfg[\"test_end\"]},\n        \"gates\": {\"min_train_positives\": cfg[\"min_train_positives\"], \"min_calibration_positives\": cfg[\"min_calibration_positives\"],\n                  \"model_vs_rule\": \"нижняя граница 95 % CI блочного бутстрэпа ΔPR-AUC (модель − правило) > 0\"},\n        \"panel\": {\"manifest_sha256\": _sha256_file(manifest_path), \"rows\": manifest.get(\"rows\"),\n                  \"schema_version\": manifest.get(\"schema_version\")},\n        \"backend_rules\": [\"при label_source = stub — плашка «ДЕМО: обучено на заглушке реестра»\",\n                          \"score разных типов и газовый score не складываются\",\n                          \"top-k объектов в сутки на тип и cooldown 72 ч применяет backend\",\n                          \"автоматических заявок нет\"],\n        \"versions\": {m: __import__(m).__version__ for m in (\"numpy\", \"polars\", \"lightgbm\", \"sklearn\")},\n    }\n    (bdir / \"bundle.json\").write_text(_dump(spec), encoding=\"utf-8\")\n    shutil.copy(MODULE_DIR / \"incident_predictor_v1.py\", bdir / \"predictor.py\")\n    (bdir / \"README_RU.md\").write_text(bundle_readme(spec), encoding=\"utf-8\")\n    sums = {p.name: _sha256_file(p) for p in sorted(bdir.iterdir()) if p.is_file()}\n    (bdir / \"SHA256SUMS.json\").write_text(json.dumps(sums, indent=2), encoding=\"utf-8\")\n    z = Path(out_dir) / \"incident_head_bundle.zip\"\n    with zipfile.ZipFile(z, \"w\", zipfile.ZIP_DEFLATED) as f:\n        for p in sorted(bdir.iterdir()):\n            f.write(p, f\"incident_head_bundle/{p.name}\")\n    (Path(out_dir) / \"incident_head_bundle.zip.sha256\").write_text(f\"{_sha256_file(z)}  incident_head_bundle.zip\\n\")\n    return bdir\n\n\ndef _public(m: Any) -> Any:\n    return None if m is None else {k: v for k, v in m.items()}\n\n\ndef bundle_readme(spec: dict[str, Any]) -> str:\n    L = [f\"# incident_head bundle — {spec['model_version']}\", \"\",\n         f\"Источник меток: **{spec['label_source']}**, уровень доказательности {spec['evidence_level']}. \"\n         \"Окно прогноза [D+2; D+3), признаки на конец D.\", \"\"]\n    if spec[\"label_source\"] == \"stub\":\n        L += [\"**ДЕМО.** Модель обучена на заглушке реестра. Цифры показывают работу конвейера и порядок числа нужных меток, \"\n              \"а не качество на реальных инцидентах.\", \"\"]\n    L += [\"| тип | статус | 1 в train / калибровке / проверке |\", \"|---|---|---|\"]\n    for k, t in spec[\"types\"].items():\n        c = t[\"counts\"]\n        L.append(f\"| {ip.TYPE_RU[k]} | {t['decision_status']} | {c.get('train_pos')} / {c.get('calibration_pos')} / {c.get('test_pos')} |\")\n    L += [\"\", \"Вызов: `b = predictor.load_bundle(dir)`; `predictor.predict(channel_records, b, as_of_date)`.\",\n          \"Контракт: docs/INCIDENT_REGISTER_CONTRACT_RU.md.\"]\n    return \"\\n\".join(L) + \"\\n\"\n\n\ndef _import_predictor(bdir: Path) -> Any:\n    import importlib.util\n\n    spec = importlib.util.spec_from_file_location(\"incident_bundle_predictor\", bdir / \"predictor.py\")\n    mod = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef acceptance(pl: Any, bdir: Path, parts: list[Path], obj: Any, fits: dict[str, Any], feats: list[str],\n               cfg: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Parity: predictor из bundle на канальных строках одних суток против модели в памяти на объектной панели.\"\"\"\n    pred = _import_predictor(bdir)\n    b = pred.load_bundle(bdir)\n    day = cfg[\"cal_end\"] + timedelta(days=30)\n    ch = pl.scan_parquet(parts).filter(pl.col(\"d_cutoff_date\") == day).collect()\n    if cfg[\"mode\"] == \"SMOKE\":\n        ch = ch.filter(pl.col(\"d_object_key\").hash(seed=7) % cfg[\"smoke_object_share\"] == 0)\n    rep: dict[str, Any] = {\"day\": day, \"objects\": int(ch[\"d_object_key\"].n_unique()) if ch.height else 0}\n    if not ch.height:\n        rep[\"parity_max_abs_diff\"] = None\n        return rep\n    out = pred.predict(ch, b, day, request_id=\"acceptance\", object_col=\"d_object_key\")\n    ref = obj.filter(pl.col(\"d_cutoff_date\") == day)\n    diffs, paths = [], {}\n    for kind, f in fits.items():\n        t = b[\"spec\"][\"types\"][kind]\n        if t[\"model_file\"] is not None:\n            mem = rt.apply_platt(_lr_obj(f[\"platt\"]), f[\"model\"].score(ref), True, 0.0)\n            paths[kind] = \"model\"\n        elif t[\"decision_status\"] == \"baseline_only\":\n            mem = rt.apply_platt(_lr_obj(f[\"rule_platt\"]), ip.rule_raw(ref, kind), False, 0.0)\n            paths[kind] = \"rule\"\n        else:\n            paths[kind] = \"abstain\"\n            continue\n        mp = dict(zip(ref[\"object_ref\"].to_list(), mem))\n        got = {r[\"d_object_key\"]: r[\"score\"] for r in out if r[\"incident_type\"] == ip.TYPE_RU[kind]}\n        diffs += [abs(got[o] - mp[o]) for o in mp if got.get(o) is not None]\n    rep[\"parity_paths\"] = paths\n    rep[\"parity_max_abs_diff\"] = float(max(diffs)) if diffs else None\n    rep[\"records\"] = len(out)\n    rep[\"has_top_factors\"] = any(r[\"top_factors\"] for r in out)\n    try:\n        pred.predict(ch.head(3), b, \"25.09.2026\")\n        rep[\"bad_date_rejected\"] = False\n    except pred.ContractError:\n        rep[\"bad_date_rejected\"] = True\n    return rep\n\n\ndef _lr_obj(p: dict[str, float]) -> Any:\n    class _M:\n        def predict_proba(self, z: np.ndarray) -> np.ndarray:\n            q = 1 / (1 + np.exp(-(p[\"coef\"] * z[:, 0] + p[\"intercept\"])))\n            return np.c_[1 - q, q]\n    return _M()\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_scenario(pl: Any, obj: Any, feats: list[str], reg: Any, coverage: dict[str, Any], cfg: dict[str, Any],\n                 log: Any, curves: bool) -> tuple[dict[str, Any], dict[str, Any]]:\n    ev, stats = load_register(pl, reg, coverage)\n    frame = build_targets(pl, obj, ev, coverage)\n    tr, ca, te = split(pl, frame, cfg)\n    fits, report = {}, {\"register\": stats, \"types\": {},\n                        \"label_reasons\": {k: {str(r): int(n) for r, n in frame[f\"reason_{k}\"].value_counts().iter_rows()}\n                                          for k in ip.GROUPS}}\n    for kind in ip.GROUPS:\n        f = fit_type(pl, tr, ca, te, kind, feats, cfg)\n        fits[kind] = f\n        r = {\"gate\": f[\"gate\"], \"decision_status\": decision_status(f[\"gate\"], coverage[\"label_source\"]),\n             \"counts\": f[\"counts\"], \"test_metrics\": f.get(\"test_metrics\"), \"importance\": f.get(\"importance\")}\n        if curves and f[\"gate\"] != \"insufficient_labels\":\n            r[\"learning_curve\"] = learning_curve(pl, tr, ca, te, kind, feats, cfg)\n        report[\"types\"][kind] = r\n        m = f.get(\"test_metrics\") or {}\n        log(f\"  {ip.TYPE_RU[kind]}: {r['decision_status']}; 1 в train/кал/тест {f['counts']['train_pos']}/\"\n            f\"{f['counts']['calibration_pos']}/{f['counts']['test_pos']}; PR-AUC модель {m.get('pr_auc_model')} \"\n            f\"правило {m.get('pr_auc_rule')} prevalence {m.get('prevalence')}\")\n    return fits, report\n\n\ndef run_incident_head(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t0 = time.time()\n    out_dir = Path(cfg[\"output_dir\"])\n    out_dir.mkdir(parents=True, exist_ok=True)\n    parts, manifest = rt.discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    manifest_path = sorted(Path(cfg[\"input_dir\"]).rglob(\"panel_manifest_v3.json\"))[0]\n    obj = build_object_panel(pl, parts, cfg)\n    feats = ip.feature_columns(obj.columns)\n    groups = (pl.scan_parquet(parts).select(\"d_channel_key\", \"тип_датчика\").unique().with_columns(ip.group_expr(pl))\n              .group_by(\"тип_датчика\", \"_grp\").agg(pl.len().alias(\"channels\")).sort(\"channels\", descending=True).collect())\n    type_groups = {str(t): {\"group\": g, \"channels\": int(n)} for t, g, n in groups.iter_rows()}\n    log(f\"объектная панель: {obj.height:,} строк объект×сутки, {obj['object_ref'].n_unique():,} объектов, признаков {len(feats)}\")\n    payload: dict[str, Any] = {\"status\": None, \"mode\": cfg[\"mode\"], \"features\": len(feats), \"object_days\": obj.height,\n                               \"objects\": int(obj[\"object_ref\"].n_unique()), \"type_groups\": type_groups, \"scenarios\": {},\n                               \"protocol\": {k: cfg[k] for k in (\"rate_per_object_year\", \"no_trace_share\", \"stub_seed\",\n                                                                \"min_train_positives\", \"min_calibration_positives\",\n                                                                \"budget_objects_per_day\", \"cooldown_days\",\n                                                                \"learning_curve_sizes\", \"train_end\", \"cal_end\", \"test_end\")}}\n    if cfg.get(\"register_path\"):\n        rp = Path(cfg[\"register_path\"])\n        reg = pl.read_parquet(rp) if rp.suffix == \".parquet\" else pl.read_csv(rp, infer_schema_length=0).with_columns(\n            pl.col(\"подтверждён\").str.to_lowercase().is_in([\"true\", \"1\", \"да\"]))\n        coverage = validate_coverage(json.loads(Path(cfg[\"coverage_path\"]).read_text(encoding=\"utf-8\")))\n        log(\"реальный реестр\")\n        main_fits, rep = run_scenario(pl, obj, feats, reg, coverage, cfg, log, curves=True)\n        payload[\"scenarios\"][\"register\"] = rep\n        main_reg, main_cov, reg_sha = reg, coverage, _sha256_file(rp)\n    else:\n        main_fits = None\n        for name, beta in cfg[\"scenarios\"].items():\n            log(f\"заглушка «{name}» (β = {beta})\")\n            reg, coverage = make_stub_register(pl, obj, cfg, beta, cfg[\"stub_seed\"])\n            fits, rep = run_scenario(pl, obj, feats, reg, coverage, cfg, log, curves=(name == cfg[\"main_scenario\"]))\n            rep[\"beta\"] = beta\n            payload[\"scenarios\"][name] = rep\n            if name == cfg[\"main_scenario\"]:\n                main_fits, main_reg, main_cov = fits, reg, coverage\n        (out_dir / \"incident_register_stub.csv\").write_bytes(main_reg.write_csv().encode(\"utf-8\"))\n        (out_dir / \"incident_register_coverage_stub.json\").write_text(_dump(main_cov), encoding=\"utf-8\")\n        reg_sha = _sha256_file(out_dir / \"incident_register_stub.csv\")\n    bdir = export_bundle(out_dir, main_fits, feats, main_cov, reg_sha, manifest, manifest_path, cfg)\n    payload[\"bundle\"] = {\"zip_sha256\": (out_dir / \"incident_head_bundle.zip.sha256\").read_text().split()[0],\n                         \"model_version\": json.loads((bdir / \"bundle.json\").read_text(encoding=\"utf-8\"))[\"model_version\"]}\n    payload[\"acceptance\"] = acceptance(pl, bdir, parts, obj, main_fits, feats, cfg)\n    payload[\"status\"] = \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\"\n    payload[\"runtime_s\"] = round(time.time() - t0, 1)\n    (out_dir / \"results_18_incident_head.json\").write_text(_dump(payload), encoding=\"utf-8\")\n    (out_dir / \"summary_18_incident_head_ru.md\").write_text(summary_markdown(payload, cfg), encoding=\"utf-8\")\n    return payload\n\n\ndef _f(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\ndef summary_markdown(p: dict[str, Any], cfg: dict[str, Any]) -> str:\n    stub = \"register\" not in p[\"scenarios\"]\n    L = [f\"# 18 · Модуль прогноза инцидентов по реестру — {p['status']}\", \"\"]\n    if stub:\n        L += [\"**ДЕМО на заглушке реестра** (evidence E4). Цифры показывают работу конвейера и порядок числа нужных меток, \"\n              \"а не качество на реальных инцидентах. Сценарий `control` (β = 0) — проверка честности: модель не должна \"\n              \"обгонять правило и случайный уровень.\", \"\"]\n    L += [f\"Объектная панель: {p['object_days']:,} объект×сутки, {p['objects']:,} объектов, {p['features']} признаков. \"\n          f\"Разбиение: train до {cfg['train_end']}, калибровка до {cfg['cal_end']}, проверка до {cfg['test_end']}. \"\n          \"2025 H2 проектом уже просматривался — это development-проверка, не независимый тест.\", \"\"]\n    tg: dict[str, list[str]] = {}\n    for t, v in p.get(\"type_groups\", {}).items():\n        tg.setdefault(v[\"group\"], []).append(f\"{t} ({v['channels']:,})\")\n    L += [\"Группы типов датчиков (по названию, заданы до запуска): \" + \"; \".join(\n        f\"**{ip.GROUP_RU.get(g, g)}** — {', '.join(v[:8])}{' …' if len(v) > 8 else ''}\" for g, v in tg.items()), \"\"]\n    k = cfg[\"budget_objects_per_day\"]\n    for name, sc in p[\"scenarios\"].items():\n        L += [f\"## {name}\" + (f\" (β = {sc['beta']})\" if \"beta\" in sc else \"\"), \"\",\n              f\"Инцидентов в реестре (в рамках ТЗ): {sc['register']['confirmed_in_scope']:,} \"\n              f\"({', '.join(f'{ip.TYPE_RU[t]} {n:,}' for t, n in sc['register']['by_type'].items())}). \"\n              \"Исключено из меток как конкурирующее событие в D+1: \"\n              + \", \".join(f\"{ip.TYPE_RU[t]} {r.get('competing_event_d1', 0):,}\" for t, r in sc[\"label_reasons\"].items()) + \".\", \"\",\n              f\"| тип | статус | 1 train/кал/тест | prevalence | PR-AUC модель | PR-AUC правило | lift модель | Δ модель−правило [95 %] | top-{k}/сутки модель: precision / recall |\",\n              \"|---|---|---|---|---|---|---|---|---|\"]\n        for kind, r in sc[\"types\"].items():\n            m, c = r.get(\"test_metrics\") or {}, r[\"counts\"]\n            b = m.get(\"bootstrap_model_minus_rule\") or {}\n            ci = f\"{_f(b.get('mean'))} [{_f(b.get('ci95_low'))}; {_f(b.get('ci95_high'))}]\" if b.get(\"mean\") is not None else \"—\"\n            tk = m.get(f\"top{k}_model\") or {}\n            L.append(f\"| {ip.TYPE_RU[kind]} | {r['decision_status']} | {c['train_pos']}/{c['calibration_pos']}/{c['test_pos']} | \"\n                     f\"{_f(m.get('prevalence'), 4)} | {_f(m.get('pr_auc_model'))} | {_f(m.get('pr_auc_rule'))} | \"\n                     f\"{_f(m.get('lift_model'), 1)} | {ci} | {_f(tk.get('precision_cooldown'))} / {_f(tk.get('recall_cooldown'))} |\")\n        curves = {kind: r[\"learning_curve\"] for kind, r in sc[\"types\"].items() if r.get(\"learning_curve\")}\n        if curves:\n            L += [\"\", \"Кривая обучения (PR-AUC модели на проверке при N положительных в train):\", \"\",\n                  \"| тип | \" + \" | \".join(f\"N = {n}\" for n in cfg[\"learning_curve_sizes\"]) + \" | все |\",\n                  \"|---|\" + \"---|\" * (len(cfg[\"learning_curve_sizes\"]) + 1)]\n            for kind, cur in curves.items():\n                byn = {x[\"n_train_positives\"]: x[\"pr_auc_model\"] for x in cur}\n                allv = (sc[\"types\"][kind].get(\"test_metrics\") or {}).get(\"pr_auc_model\")\n                L.append(f\"| {ip.TYPE_RU[kind]} | \" + \" | \".join(_f(byn.get(n)) for n in cfg[\"learning_curve_sizes\"])\n                         + f\" | {_f(allv)} |\")\n        L.append(\"\")\n    a = p[\"acceptance\"]\n    L += [\"## Приёмка bundle\", \"\",\n          f\"- `{p['bundle']['model_version']}`, ZIP sha256 `{p['bundle']['zip_sha256'][:12]}…`.\",\n          f\"- Parity predictor из bundle ↔ расчёт в памяти на сутках {a.get('day')} (пути: {a.get('parity_paths')}): max |Δ| = {a.get('parity_max_abs_diff')}.\",\n          f\"- top_factors есть: {a.get('has_top_factors')}; неверная дата отклоняется: {a.get('bad_date_rejected')}.\", \"\"]\n    return \"\\n\".join(L)\n\n\n# ---------------------------------------------------------------------------\n# Синтетическая канальная панель для тестов (не реальные данные)\n# ---------------------------------------------------------------------------\nSYNTH_TYPES = [\"Газовый датчик\", \"Датчик дыма\", \"Датчик затопления\", \"Состояние насоса\", \"Датчик движения\", \"КД Дверь\"]\n\n\ndef synthetic_incident_panel(out_dir: Path, n_objects: int = 60, seed: int = 5,\n                             start: date = date(2022, 1, 1), end: date = date(2026, 1, 10)) -> Path:\n    \"\"\"Суточная панель v3-формы: у объекта 6 каналов; активность тревог — AR(1) по логарифму интенсивности,\n    поэтому всплеск в D+2 частично предсказуем по истории. Для тестов модуля 18.\"\"\"\n    import polars as pl\n\n    rng = np.random.default_rng(seed)\n    days = np.arange(np.datetime64(start), np.datetime64(end))\n    n_days, n_ch = len(days), n_objects * len(SYNTH_TYPES)\n    lam = np.zeros(n_ch)\n    alarms = np.zeros((n_days, n_ch))\n    season = 1 + 0.8 * np.sin(2 * np.pi * (days.astype(\"datetime64[D]\").astype(int) % 365) / 365)\n    is_flood = np.tile(np.isin(SYNTH_TYPES, [\"Датчик затопления\", \"Состояние насоса\"]), n_objects)\n    for t in range(n_days):\n        lam = 0.92 * lam + rng.normal(0, 0.35, n_ch)\n        rate = 0.3 * np.exp(lam) * np.where(is_flood, season[t], 1.0)\n        alarms[t] = rng.poisson(rate)\n    csum = np.vstack([np.zeros((1, n_ch)), np.cumsum(alarms, axis=0)])\n    idx = np.arange(n_days)\n    a7 = csum[idx + 1] - csum[np.maximum(idx - 6, 0)]\n    a30 = csum[idx + 1] - csum[np.maximum(idx - 29, 0)]\n    ch = np.tile(np.arange(n_ch), n_days)\n    obj = ch // len(SYNTH_TYPES)\n    typ = np.array(SYNTH_TYPES)[ch % len(SYNTH_TYPES)]\n    n = n_days * n_ch\n    gas = np.where(typ == \"Газовый датчик\", rng.uniform(0, 0.8, n) + (alarms.ravel() > 2) * 0.6, np.nan)\n    frame = {\n        \"d_channel_key\": np.array([hashlib.sha256(f\"c{c}\".encode()).hexdigest()[:16] for c in range(n_ch)])[ch],\n        \"d_object_key\": np.array([hashlib.sha256(f\"o{o}\".encode()).hexdigest()[:16] for o in range(n_objects)])[obj],\n        \"d_cutoff_date\": np.repeat(days, n_ch), \"тип_датчика\": typ,\n        \"link_state\": rng.choice([\"O\", \"F\", \"A\"], n, p=[0.93, 0.04, 0.03]),\n        \"power_state\": rng.choice([\"O\", \"F\"], n, p=[0.95, 0.05]),\n        \"n_alarm\": alarms.ravel(), \"n_events\": alarms.ravel() + rng.poisson(20, n),\n        \"f_n_alarm_7d\": a7.ravel(), \"f_n_alarm_30d\": a30.ravel(), \"f_n_events_7d\": a7.ravel() + rng.poisson(140, n),\n        \"f_n_flood_7d\": np.where(is_flood[ch], a7.ravel(), 0.0), \"f_n_link_fault_7d\": rng.poisson(0.2, n).astype(float),\n        \"gas_max\": gas, \"f_gas_ge_thr_n_7d\": np.where(np.nan_to_num(gas) >= 1, 1.0, 0.0),\n    }\n    panel = pl.DataFrame(frame).with_columns(pl.col(\"d_cutoff_date\").cast(pl.Date))\n    panel_dir = Path(out_dir) / \"event_panel_v3\"\n    panel_dir.mkdir(parents=True, exist_ok=True)\n    part = panel_dir / \"event_panel_v3_part00.parquet\"\n    panel.write_parquet(part)\n    manifest = {\"schema_version\": rt.SCHEMA_VERSION, \"rows\": panel.height, \"taxonomy_sha256\": \"synthetic\",\n                \"panel_config\": {\"synthetic\": True, \"module\": \"18\"}, \"feature_columns\": [c for c in panel.columns if c.startswith(\"f_\")],\n                \"target_columns\": [], \"parts\": [{\"file\": part.name, \"sha256\": rt.file_sha256(part), \"rows\": panel.height}]}\n    (panel_dir / \"panel_manifest_v3.json\").write_text(json.dumps(manifest), encoding=\"utf-8\")\n    return panel_dir\n\n\n# ---------------------------------------------------------------------------\n# CLI (дообучение на реальном реестре)\n# ---------------------------------------------------------------------------\ndef main(argv: list[str] | None = None) -> int:\n    ap = argparse.ArgumentParser(description=\"incident_head: обучение по реестру инцидентов\")\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n    t = sub.add_parser(\"train\")\n    t.add_argument(\"--panel\", required=True, help=\"каталог с выходом ноутбука 14 (panel_manifest_v3.json)\")\n    t.add_argument(\"--register\", help=\"incident_register.csv|.parquet; без него — заглушка\")\n    t.add_argument(\"--coverage\", help=\"incident_register_coverage.json\")\n    t.add_argument(\"--out\", required=True)\n    t.add_argument(\"--mode\", default=\"FULL\", choices=[\"FULL\", \"SMOKE\"])\n    a = ap.parse_args(argv)\n    if bool(a.register) != bool(a.coverage):\n        ap.error(\"реестр и покрытие передаются вместе\")\n    cfg = make_config_18(a.mode, {\"input_dir\": a.panel, \"output_dir\": a.out, \"register_path\": a.register,\n                                  \"coverage_path\": a.coverage})\n    res = run_incident_head(cfg)\n    print(json.dumps({k: res[\"scenarios\"][s][\"types\"][k][\"decision_status\"] for s in res[\"scenarios\"]\n                      for k in res[\"scenarios\"][s][\"types\"]}, ensure_ascii=False))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
}


In [ ]:
import tempfile
from pathlib import Path
CODE_DIR = Path(tempfile.mkdtemp(prefix="ldt18_code_"))
for name, text in SOURCES.items():
    (CODE_DIR / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(CODE_DIR))
import incident_head_v1 as ih
print("код записан:", ", ".join(sorted(SOURCES)))


## Прогон


In [ ]:
CONFIG = ih.make_config_18(MODE)
RESULT = ih.run_incident_head(CONFIG)
print("Статус:", RESULT["status"], "| время:", RESULT["runtime_s"], "с")


## Сводка


In [ ]:
from IPython.display import Markdown, display
OUT = Path(CONFIG["output_dir"])
display(Markdown((OUT / "summary_18_incident_head_ru.md").read_text(encoding="utf-8")))
for name in ("results_18_incident_head.json", "summary_18_incident_head_ru.md", "incident_head_bundle.zip",
             "incident_head_bundle.zip.sha256", "incident_register_stub.csv", "incident_register_coverage_stub.json"):
    p = OUT / name
    print("-", name, f"({p.stat().st_size / 1024:.0f} КБ)" if p.exists() else "— НЕ СОЗДАН")
